# Notebook 21 — Comment and Embedded Information

## Bounded question

What does the runner-level `comment` field contain, how consistently is it populated and structured, which information duplicates existing structured fields, which genuinely new information can be extracted deterministically, and what must remain preserved as source free text?

## Purpose

This notebook investigates the physical behaviour, coverage, structure and meaning of the source `comment` field.

The study will not assume that comments are globally standardised, wholly objective, independently authored, or safe to parse into structured facts.

It will distinguish between:

* immutable source text;
* exact deterministic extraction;
* heuristic text classification;
* interpretive racing analysis;
* publication-safe derived summaries.

The raw `comment` value will be preserved unchanged. Any proposed extraction must be evidence-led, reversible, explicit about uncertainty and independently validatable across the relevant source population.

## Source and grain

* **Source database:** `data/raw/form_2015-present/raceform.db`
* **Source table:** `data`
* **Governed data-row predicate:** `rowid <> 1`
* **Field under investigation:** `comment`
* **Declared source type:** `TEXT`
* **Provisional grain:** runner-level source assertion
* **Raw preservation:** required
* **Current governance status:** pending semantics

The apparent runner-level grain must be tested rather than assumed. The field may contain runner-specific observations, repeated race-level text, templated source language, or a mixture of these.

## Initial scope

The investigation will establish:

* physical storage behaviour;
* null, empty and whitespace-only values;
* coverage by period, jurisdiction, course and race type;
* exact-text repetition and likely templates;
* punctuation, symbols, abbreviations and source conventions;
* whether comments are runner-level, race-level or mixed;
* embedded finishing-position and in-running information;
* incidents, interference and jumping errors;
* pace, effort and performance language;
* equipment references;
* information duplicated in existing structured fields;
* genuinely additional information;
* deterministic and reversible extraction candidates;
* ambiguous or interpretive cases that must remain free text;
* publication and licensing constraints on examples and derived outputs.

No reusable parser, classification system or permanent reference output is authorised in advance. Stable implementation will be extracted only where the notebook evidence justifies it.


## 1. Establish the source population and physical field behaviour

The first stage will confirm the source population, inspect the declared SQLite storage for `comment`, and measure its basic missingness and text behaviour.

This stage will establish:

* the governed runner-row count;
* the SQLite declared type for `comment`;
* null values;
* empty strings;
* whitespace-only strings;
* leading or trailing whitespace;
* populated values;
* distinct populated values;
* minimum, maximum and typical text lengths.

These checks come before interpretation. A populated text field can still contain placeholders, duplicated templates, formatting artefacts or inconsistent source conventions.

No comment text will be normalised or overwritten during this profiling stage.


In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
SOURCE_DB_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

DATA_ROW_PREDICATE = "rowid <> 1"

assert SOURCE_DB_PATH.exists(), f"Source database not found: {SOURCE_DB_PATH}"

connection = sqlite3.connect(f"file:{SOURCE_DB_PATH}?mode=ro", uri=True)

schema = pd.read_sql_query("PRAGMA table_info(data)", connection)
comment_schema = schema.loc[schema["name"].eq("comment")].copy()

comment_profile = pd.read_sql_query(
    f"""
    SELECT
        COUNT(*) AS governed_runner_rows,
        SUM(comment IS NULL) AS null_rows,
        SUM(comment = '') AS empty_string_rows,
        SUM(
            comment IS NOT NULL
            AND comment <> ''
            AND TRIM(comment) = ''
        ) AS whitespace_only_rows,
        SUM(
            comment IS NOT NULL
            AND TRIM(comment) <> ''
        ) AS populated_rows,
        COUNT(
            DISTINCT CASE
                WHEN comment IS NOT NULL AND TRIM(comment) <> ''
                THEN comment
            END
        ) AS distinct_populated_values,
        SUM(
            comment IS NOT NULL
            AND comment <> LTRIM(comment)
        ) AS leading_whitespace_rows,
        SUM(
            comment IS NOT NULL
            AND comment <> RTRIM(comment)
        ) AS trailing_whitespace_rows,
        MIN(
            CASE
                WHEN comment IS NOT NULL AND TRIM(comment) <> ''
                THEN LENGTH(comment)
            END
        ) AS minimum_populated_length,
        MAX(
            CASE
                WHEN comment IS NOT NULL AND TRIM(comment) <> ''
                THEN LENGTH(comment)
            END
        ) AS maximum_populated_length
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

display(comment_schema)
display(comment_profile)

,cid,name,type,notnull,dflt_value,pk
36,36,comment,TEXT,0,None,0


,governed_runner_rows,null_rows,empty_string_rows,whitespace_only_rows,populated_rows,distinct_populated_values,leading_whitespace_rows,trailing_whitespace_rows,minimum_populated_length,maximum_populated_length
0,1851285,0,340394,0,1510891,1426745,1,0,1,2206


### Initial physical findings

The governed source contains **1,851,285 runner rows**.

For `comment`:

* **0** values are SQL `NULL`;
* **340,394** values are empty strings;
* **0** values contain whitespace only;
* **1,510,891** values contain populated text;
* **1,426,745** distinct populated strings occur;
* populated comment length ranges from **1** to **2,206** characters;
* **1** populated value has leading whitespace;
* **0** populated values have trailing whitespace.

The source therefore represents missing comments as empty text rather than database nulls. For this field, the empty string is a source-level absence representation and must not be treated as meaningful commentary.

The very high number of distinct populated values suggests that most comments are not exact copies of a small template vocabulary. However, the difference between populated rows and distinct populated strings shows that exact repetition does occur and requires separate profiling.

The single leading-whitespace case is an anomaly to inspect rather than silently trim. The maximum length of 2,206 characters is also sufficiently unusual to require direct review before assumptions are made about the normal comment structure.

At this stage:

* raw text remains unchanged;
* empty string is provisionally classified as `field_not_supplied`;
* populated comments remain unclassified source assertions;
* no normalisation or extraction rule is authorised.


## 2. Profile comment length and exact-text repetition

The next stage will test whether the populated comments form a broadly narrative field or contain substantial repeated templates and unusual structural outliers.

The analysis will examine:

* comment-length distribution;
* shortest populated comments;
* longest populated comments;
* frequency of exact repeated strings;
* the most common repeated comments;
* how many populated rows belong to unique versus repeated text;
* the single leading-whitespace anomaly.

This stage remains descriptive. Repetition does not by itself prove that a phrase is a formal template, and unusual length does not by itself prove corruption.

Direct examples will be limited to small investigative samples. The full source text will not be reproduced or exported.


In [2]:
# Profile the distribution of populated comment lengths.
# The bins are descriptive only and do not imply semantic categories.
length_profile = pd.read_sql_query(
    f"""
    SELECT
        COUNT(*) AS populated_rows,
        ROUND(AVG(LENGTH(comment)), 2) AS mean_length,
        MIN(LENGTH(comment)) AS minimum_length,
        MAX(LENGTH(comment)) AS maximum_length,
        SUM(LENGTH(comment) <= 10) AS length_1_to_10,
        SUM(LENGTH(comment) BETWEEN 11 AND 25) AS length_11_to_25,
        SUM(LENGTH(comment) BETWEEN 26 AND 50) AS length_26_to_50,
        SUM(LENGTH(comment) BETWEEN 51 AND 100) AS length_51_to_100,
        SUM(LENGTH(comment) BETWEEN 101 AND 250) AS length_101_to_250,
        SUM(LENGTH(comment) BETWEEN 251 AND 500) AS length_251_to_500,
        SUM(LENGTH(comment) > 500) AS length_over_500
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND comment <> ''
    """,
    connection,
)

# Count distinct exact strings and separate comments that occur once
# from comments that are repeated exactly elsewhere in the source.
repetition_profile = pd.read_sql_query(
    f"""
    WITH comment_counts AS (
        SELECT
            comment,
            COUNT(*) AS occurrence_count
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND comment <> ''
        GROUP BY comment
    )
    SELECT
        SUM(occurrence_count = 1) AS unique_text_values,
        SUM(occurrence_count > 1) AS repeated_text_values,
        SUM(
            CASE
                WHEN occurrence_count = 1
                THEN occurrence_count
                ELSE 0
            END
        ) AS rows_with_unique_text,
        SUM(
            CASE
                WHEN occurrence_count > 1
                THEN occurrence_count
                ELSE 0
            END
        ) AS rows_with_repeated_text,
        MAX(occurrence_count) AS maximum_exact_repetition
    FROM comment_counts
    """,
    connection,
)

# Inspect the most frequently repeated exact strings.
# These examples may reveal templates, placeholders or highly generic phrases,
# but repetition alone will not be treated as proof of a formal template.
most_repeated_comments = pd.read_sql_query(
    f"""
    SELECT
        comment,
        COUNT(*) AS occurrence_count,
        LENGTH(comment) AS comment_length
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND comment <> ''
    GROUP BY comment
    HAVING COUNT(*) > 1
    ORDER BY occurrence_count DESC, comment
    LIMIT 20
    """,
    connection,
)

# Inspect the shortest populated comments with source lineage.
# Very short values may be meaningful abbreviations, punctuation artefacts,
# placeholders or malformed source text.
shortest_comments = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        horse,
        comment,
        LENGTH(comment) AS comment_length
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND comment <> ''
    ORDER BY LENGTH(comment), rowid
    LIMIT 20
    """,
    connection,
)

# Inspect the longest comments with source lineage.
# These cases may expose concatenated text, unusual source coverage,
# embedded reports or other departures from the normal field structure.
longest_comments = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        horse,
        LENGTH(comment) AS comment_length,
        comment
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND comment <> ''
    ORDER BY LENGTH(comment) DESC, rowid
    LIMIT 10
    """,
    connection,
)

# Retrieve the single leading-whitespace anomaly without modifying it.
# quote(comment) makes the exact stored boundary visible in the output.
leading_whitespace_case = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        horse,
        LENGTH(comment) AS comment_length,
        quote(comment) AS quoted_comment
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND comment <> LTRIM(comment)
    """,
    connection,
)

display(length_profile)
display(repetition_profile)
display(most_repeated_comments)
display(shortest_comments)
display(longest_comments)
display(leading_whitespace_case)

,populated_rows,mean_length,minimum_length,maximum_length,length_1_to_10,length_11_to_25,length_26_to_50,length_51_to_100,length_101_to_250,length_251_to_500,length_over_500
0,1510891,121.98,1,2206,304,15262,69787,483933,912648,27767,1190


,unique_text_values,repeated_text_values,rows_with_unique_text,rows_with_repeated_text,maximum_exact_repetition
0,1405900,20845,1405900,104991,4547


,comment,occurrence_count,comment_length
0,Always towards rear,4547,19
1,Never better than mid-division,2169,30
2,Always in rear,1772,14
3,Never near to challenge,981,23
4,Always behind,652,13
5,Slowly into stride - always in rear,519,35
6,Never better than midfield,516,26
7,Always mid-division,484,19
8,Slowly into stride - never near to challenge,370,44
9,Towards rear throughout,342,23


,source_rowid,date,course,off,horse,comment,comment_length
0,29974,2015-03-28,Nakayama (JPN),6:45,Admire Deus (JPN),B,1
1,137122,2015-11-03,Chantilly (FR),2:55,Black Bird Runs (FR),.,1
2,147355,2015-11-29,Kyoto (JPN),7:15,Satono Lupin (JPN),.,1
3,171731,2016-02-20,Rosehill (AUS),4:05,First Seal (AUS),B,1
4,240574,2016-07-18,Chantilly (FR),2:20,Aladdine (GB),A,1
5,366454,2017-05-05,Auteuil (FR),12:40,Darling Des Bordes (FR),.,1
6,487574,2018-01-13,Gulfstream Park (USA),9:00,Ultra Brat (USA),B,1
7,494222,2018-02-03,Caulfield (AUS),5:45,Cliffs Edge (AUS),.,1
8,494223,2018-02-03,Caulfield (AUS),5:45,Overshare (AUS),.,1
9,494225,2018-02-03,Caulfield (AUS),5:45,I Did It Again (AUS),.,1


,source_rowid,date,course,off,horse,comment_length,comment
0,1622460,2025-01-13,Wolverhampton (AW),5:30,Further Measure (USA),2206,Raced in last - still plenty to do when switch...
1,1745480,2025-10-06,Wolverhampton (AW),4:20,Oman (IRE),2083,In rear and raced wide early - still plenty to...
2,873883,2020-08-13,Tramore (IRE),6:15,Enzani (IRE),2034,Mid-division - dropped towards rear 6th - deta...
3,1337070,2023-05-10,Gowran Park (IRE),5:25,Ellaat (GB),2023,Dwelt start - towards rear - headway under 3f ...
4,1753197,2025-10-20,Wolverhampton (AW),16:55,Bandello (GB),1990,In rear - still plenty to do 2f out - headway ...
5,1478171,2024-03-19,Wolverhampton (AW),7:30,Kodi Noir (IRE),1986,Slowly away - in rear - no telling impression ...
6,1653708,2025-03-31,Wolverhampton (AW),4:55,Nevernay (IRE),1948,In rear - awkward start and lost many lengths ...
7,1140782,2022-02-10,Doncaster,3:20,Flaming Ambition (IRE),1937,Took keen hold - held up in rear - nudged alon...
8,1657172,2025-04-07,Wolverhampton (AW),4:05,Charlatan (IRE),1896,Held up in rear - not clear run over 1f out - ...
9,1434551,2023-11-25,Huntingdon,12:28,Zain Nights (GB),1895,Midfield - not fluent 3 out - soon shaken up -...


,source_rowid,date,course,off,horse,comment_length,quoted_comment
0,1833016,2026-04-24,Bahrain,16:00,Salamanca Lad (IRE),2,' -'


### Length and exact-repetition findings

The populated `comment` field is predominantly narrative rather than a small set of repeated templates.

Across **1,510,891 populated rows**:

* mean comment length is **121.98 characters**;
* **912,648** comments are between 101 and 250 characters;
* **483,933** are between 51 and 100 characters;
* only **304** contain 10 characters or fewer;
* **1,190** exceed 500 characters;
* the maximum length is **2,206 characters**.

Exact repetition is limited but material:

* **1,405,900 populated rows**, or approximately **93.05%**, contain text that occurs only once;
* **104,991 rows**, or approximately **6.95%**, belong to an exactly repeated string;
* **20,845 distinct strings** occur more than once;
* the most repeated exact phrase occurs **4,547 times**.

The most frequent repeated values are short, generic in-running summaries such as rear-position or non-challenge descriptions. This is consistent with reusable source phrasing, but does not yet establish a formal template system. Some repeated strings also contain appended starting-price material, showing that apparently similar commentary may combine narrative and structured-looking suffixes.

The shortest populated values reveal several distinct possibilities:

* `"."` appears as a one-character placeholder rather than meaningful commentary;
* single letters such as `"A"` and `"B"` may be source codes, contamination, or jurisdiction-specific notation;
* their meaning cannot be inferred from length alone.

The single leading-whitespace case contains the exact two-character value `" -"`. This appears more likely to be another absence or placeholder representation than substantive commentary, but it must be investigated alongside the other shortest values before the blank policy is finalised.

The longest comments are not random isolated characters. Their visible beginnings resemble ordinary in-running narratives, suggesting possible concatenation or repeated appended material rather than an unrelated text type. Their full internal structure must be examined before they are classified as valid long reports or source corruption.

Current provisional conclusions:

* most populated comments are unique runner-level narratives;
* a smaller repeated vocabulary exists and may contain generic templates;
* `"."`, `" -"`, and isolated letters require targeted investigation;
* very long comments require structural inspection;
* no text-length trimming, placeholder conversion or extraction rule is yet authorised.


## 3. Investigate short values and probable placeholders

The shortest populated comments contain values that may not represent ordinary in-running narratives.

External research indicates that the field is generally a runner-level close-up comment: compressed chronological prose describing a particular performance. However, no standard close-up-comment convention has been identified that explains standalone values such as `"A"`, `"B"`, `"."` or `" -"`.

These values must therefore be investigated from the source rather than assigned meanings from unrelated racing abbreviations.

This stage will examine:

* every distinct populated comment of ten characters or fewer;
* frequency by exact value;
* first and last occurrence;
* course and jurisdiction distribution;
* whether values are concentrated within particular races or source periods;
* the surrounding comments from affected races;
* whether `"."`, `" -"`, isolated letters and symbols behave like missing-value placeholders;
* whether any short values are legitimate compressed racing descriptions.

The following distinctions will be preserved:

* empty source string;
* punctuation-only value;
* whitespace-and-punctuation value;
* isolated letter;
* short word or abbreviation;
* ordinary short narrative.

A familiar abbreviation used elsewhere in racing must not automatically be transferred into this field. For example, `"B"` can have a documented meaning in result or form notation, but that does not establish the same meaning when it appears as the complete `comment` value.

No short value will be converted to null, expanded into a meaning or otherwise normalised until its source distribution and surrounding race context have been inspected.


In [3]:
# Profile every populated comment containing ten characters or fewer.
# Exact raw values are preserved with quote(comment) so that spaces,
# punctuation and other boundary characters remain visible.
short_value_summary = pd.read_sql_query(
    f"""
    SELECT
        quote(comment) AS quoted_comment,
        LENGTH(comment) AS comment_length,
        COUNT(*) AS runner_rows,
        COUNT(
            DISTINCT date || '|' || course || '|' || off
        ) AS provisional_races,
        COUNT(DISTINCT course) AS distinct_courses,
        MIN(date) AS first_date,
        MAX(date) AS last_date
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND comment <> ''
      AND LENGTH(comment) <= 10
    GROUP BY comment
    ORDER BY
        runner_rows DESC,
        comment_length,
        quoted_comment
    """,
    connection,
)

# Retain a small lineage-bearing sample for every exact short value.
# These examples are for contextual inspection only; they do not assign
# meanings or convert any value into a missing-value category.
short_value_examples = pd.read_sql_query(
    f"""
    WITH ranked_examples AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            horse,
            quote(comment) AS quoted_comment,
            LENGTH(comment) AS comment_length,
            ROW_NUMBER() OVER (
                PARTITION BY comment
                ORDER BY date, course, off, rowid
            ) AS example_number
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND comment <> ''
          AND LENGTH(comment) <= 10
    )
    SELECT
        source_rowid,
        date,
        course,
        off,
        horse,
        quoted_comment,
        comment_length
    FROM ranked_examples
    WHERE example_number <= 5
    ORDER BY
        comment_length,
        quoted_comment,
        date,
        course,
        off,
        source_rowid
    """,
    connection,
)

display(short_value_summary)
display(short_value_examples)

,quoted_comment,comment_length,runner_rows,provisional_races,distinct_courses,first_date,last_date
0,'.',1,225,49,21,2015-11-03,2026-05-23
1,'Fell 1st',8,43,37,23,2015-01-29,2019-12-30
2,'B',1,4,4,4,2015-03-28,2020-12-19
3,'/',1,2,1,1,2026-04-24,2026-04-24
4,'A',1,2,2,2,2016-07-18,2021-04-07
5,'(op 6/1)',8,2,2,2,2017-09-23,2025-04-11
6,'(Op. 10/1)',10,2,1,1,2018-10-07,2018-10-07
7,'(Op. 14/1)',10,2,1,1,2018-10-07,2018-10-07
8,'-',1,1,1,1,2019-12-12,2019-12-12
9,'1',1,1,1,1,2021-10-17,2021-10-17


,source_rowid,date,course,off,horse,quoted_comment,comment_length
0,820825,2019-12-12,Chantilly (FR),11:40,Shayandi (FR),'-',1
1,137122,2015-11-03,Chantilly (FR),2:55,Black Bird Runs (FR),'.',1
2,147355,2015-11-29,Kyoto (JPN),7:15,Satono Lupin (JPN),'.',1
3,366454,2017-05-05,Auteuil (FR),12:40,Darling Des Bordes (FR),'.',1
4,494222,2018-02-03,Caulfield (AUS),5:45,Cliffs Edge (AUS),'.',1
5,494223,2018-02-03,Caulfield (AUS),5:45,Overshare (AUS),'.',1
6,1833023,2026-04-24,Bahrain,17:30,Cover Point (GB),'/',1
7,1833027,2026-04-24,Bahrain,17:30,Mr Irrelevant (IRE),'/',1
8,1091089,2021-10-17,Longchamp (FR),12:58,Epic Poet (IRE),'1',1
9,240574,2016-07-18,Chantilly (FR),2:20,Aladdine (GB),'A',1


### Short-value findings

The **304 populated comments of ten characters or fewer** divide into three materially different groups rather than one general anomaly class.

#### 1. Probable placeholders or unresolved source codes

There are **238 rows** containing punctuation-only, numeric-only or isolated-letter values:

* `"."`: **225 rows**, across 49 provisional races and 21 courses;
* `"B"`: **4 rows**, each in a different race, course and jurisdiction;
* `"A"`: **2 rows**, in two French races;
* `"/"`: **2 rows**, both from one Bahrain race;
* `"-"`: **1 row**;
* `"1"`: **1 row**;
* `"V"`: **1 row**;
* `" -"`: **1 row**;
* `".."`: **1 row**.

These values account for approximately **78.3%** of all comments containing ten characters or fewer, but only about **0.0158%** of all populated comments.

The dominant anomaly is `"."`. It occurs across multiple jurisdictions and years and can affect several runners within the same race. The Racing Post result display reproduces the dots directly beneath affected runners, confirming that they are not introduced by this database extraction. Their repeated placement where an ordinary close-up comment would appear strongly suggests a published placeholder for unavailable commentary, although the exact upstream convention remains undocumented.

The isolated letters are exceptionally rare:

* `"B"` occurs four times between 2015 and 2020 in Japan, Australia, the United States and Argentina;
* `"A"` occurs twice, in France in 2016 and 2021;
* `"V"` occurs once in Argentina.

Original Racing Post result pages also reproduce at least the inspected `"A"` and `"B"` values. They are therefore source-presented values rather than database corruption. However, no reliable published convention has yet been found that establishes their meaning in the close-up-comment field. They must remain unresolved source codes rather than being expanded from similarly named abbreviations used elsewhere in racing.

The two slashes and the `" -"` value are concentrated in Bahrain on 24 April 2026 and may reflect a recent feed-specific absence convention. The single dash, double dot and numeric `"1"` require the same cautious treatment.

#### 2. Legitimate short close-up comments

There are **49 rows** containing plausible substantive racing commentary:

* `"Fell 1st"`: **43 rows**;
* `"In rear"`: **1 row**;
* `"Midfield"`: **1 row**;
* `"Poor run"`: **1 row**;
* `"Walkover"`: **1 row**;
* `"No finish"`: **1 row**;
* `"Always 4th"`: **1 row**.

This confirms that length alone cannot identify missing or malformed commentary. Some valid and analytically meaningful descriptions are extremely short.

`"Fell 1st"` is the clearest example: it is concise but records both a non-completion event and the obstacle at which it occurred. It may duplicate or enrich structured result information, but it is not an absence marker.

#### 3. Betting-market text without an in-running narrative

There are **17 rows** whose complete comment consists only of an opening-price parenthetical, such as:

* `"(op 6/1)"`;
* `"(op 11/8)"`;
* `"(Op. 10/1)"`.

Several are concentrated within particular races, especially Fontwell on 11 April 2025 and Longchamp on 7 October 2018.

These rows show that the field can contain structured-looking betting history even when no in-running narrative is supplied. They should not be classified as empty comments, but they also demonstrate that `comment` is not a homogeneous narrative field.

Capitalisation and punctuation vary between forms such as `op` and `Op.`, indicating either period-specific or feed-specific formatting differences.

### Current classification

The short-value evidence supports the following provisional source states:

* `empty_string`: no text supplied;
* `probable_placeholder`: punctuation-only values such as `"."`, `"-"`, `".."`, `"/"` and `" -"`;
* `unresolved_source_code`: isolated letters or numeric values such as `"A"`, `"B"`, `"V"` and `"1"`;
* `market_only_comment`: opening-price text with no narrative;
* `substantive_short_comment`: valid compact race description.

These are analytical classifications layered over the raw value. The source text must remain unchanged.

No automatic null conversion is yet authorised. In particular:

* `"."` is strongly consistent with a placeholder but remains distinct from the empty string;
* isolated letters remain unresolved;
* short substantive comments must not be lost through a length-based rule;
* odds-only comments contain genuine information even when narrative commentary is absent.


## 4. Test the equipment-change hypothesis

A specialist form reader suggested that the isolated letters may indicate a gear change, such as blinkers or a visor.

He did not recognise the letters as an established Raceform or Racing Post close-up-comment convention, so this remains a hypothesis rather than external validation.

This stage will test the idea directly against the source by examining:

- the structured `hg` value on every runner whose complete comment is `A`, `B` or `V`;
- the same horse’s immediately preceding and following source appearances;
- whether headgear is present on the coded run;
- whether the coded run coincides with a change in the recorded equipment value;
- whether one letter consistently corresponds to one equipment state.

The test is deliberately bounded. It will not reopen the full semantics of the `hg` field or attempt to reconstruct equipment history beyond the adjacent source appearances needed to evaluate this hypothesis.

Possible outcomes are:

- **supported**: a stable and repeated relationship is visible;
- **partially supported**: some coded rows coincide with equipment changes but the pattern is inconsistent;
- **not supported**: the letters do not align with recorded equipment;
- **unresolved**: source coverage is insufficient to test the idea safely.

The raw `comment` and `hg` values will remain unchanged regardless of the result.

In [4]:
# Inspect every runner whose complete comment is one of the isolated
# letters under investigation.
#
# The source `hg` value is preserved exactly as stored. Adjacent appearances
# are identified by horse label and source chronology only; they are not
# assumed to prove stable real-world horse identity beyond the source label.
letter_code_equipment_context = pd.read_sql_query(
    f"""
    WITH ordered_horse_runs AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            horse,
            hg,
            comment,

            LAG(date) OVER (
                PARTITION BY horse
                ORDER BY date, rowid
            ) AS previous_date,

            LAG(course) OVER (
                PARTITION BY horse
                ORDER BY date, rowid
            ) AS previous_course,

            LAG(off) OVER (
                PARTITION BY horse
                ORDER BY date, rowid
            ) AS previous_off,

            LAG(hg) OVER (
                PARTITION BY horse
                ORDER BY date, rowid
            ) AS previous_hg,

            LAG(comment) OVER (
                PARTITION BY horse
                ORDER BY date, rowid
            ) AS previous_comment,

            LEAD(date) OVER (
                PARTITION BY horse
                ORDER BY date, rowid
            ) AS next_date,

            LEAD(course) OVER (
                PARTITION BY horse
                ORDER BY date, rowid
            ) AS next_course,

            LEAD(off) OVER (
                PARTITION BY horse
                ORDER BY date, rowid
            ) AS next_off,

            LEAD(hg) OVER (
                PARTITION BY horse
                ORDER BY date, rowid
            ) AS next_hg,

            LEAD(comment) OVER (
                PARTITION BY horse
                ORDER BY date, rowid
            ) AS next_comment
        FROM data
        WHERE {DATA_ROW_PREDICATE}
    )
    SELECT
        source_rowid,
        date,
        course,
        off,
        horse,
        quote(comment) AS quoted_comment,
        quote(hg) AS quoted_hg,

        previous_date,
        previous_course,
        previous_off,
        quote(previous_hg) AS quoted_previous_hg,
        previous_comment,

        next_date,
        next_course,
        next_off,
        quote(next_hg) AS quoted_next_hg,
        next_comment
    FROM ordered_horse_runs
    WHERE comment IN ('A', 'B', 'V')
    ORDER BY date, course, off, horse
    """,
    connection,
)

# Summarise whether the coded run's equipment value differs from the
# immediately preceding or following source appearance.
#
# A difference is descriptive only. It does not establish that the letter
# means an equipment change, because blank values, jurisdiction conventions
# and unrelated source changes may also affect `hg`.
letter_code_equipment_summary = pd.read_sql_query(
    f"""
    WITH ordered_horse_runs AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            horse,
            hg,
            comment,

            LAG(hg) OVER (
                PARTITION BY horse
                ORDER BY date, rowid
            ) AS previous_hg,

            LEAD(hg) OVER (
                PARTITION BY horse
                ORDER BY date, rowid
            ) AS next_hg
        FROM data
        WHERE {DATA_ROW_PREDICATE}
    )
    SELECT
        comment AS letter_code,
        COUNT(*) AS coded_runner_rows,

        SUM(hg IS NULL) AS current_hg_null_rows,
        SUM(hg = '') AS current_hg_empty_rows,
        SUM(hg IS NOT NULL AND hg <> '') AS current_hg_populated_rows,

        SUM(previous_hg IS NOT NULL) AS rows_with_previous_run,
        SUM(next_hg IS NOT NULL) AS rows_with_next_run,

        SUM(
            previous_hg IS NOT NULL
            AND hg IS NOT previous_hg
        ) AS differs_from_previous_hg,

        SUM(
            next_hg IS NOT NULL
            AND hg IS NOT next_hg
        ) AS differs_from_next_hg
    FROM ordered_horse_runs
    WHERE comment IN ('A', 'B', 'V')
    GROUP BY comment
    ORDER BY comment
    """,
    connection,
)

display(letter_code_equipment_context)
display(letter_code_equipment_summary)

,source_rowid,date,course,off,horse,quoted_comment,quoted_hg,previous_date,previous_course,previous_off,quoted_previous_hg,previous_comment,next_date,next_course,next_off,quoted_next_hg,next_comment
0,29974,2015-03-28,Nakayama (JPN),6:45,Admire Deus (JPN),'B','',2015-01-18,Kyoto (JPN),6:45,'',,2015-05-03,Kyoto (JPN),7:40,'',
1,171731,2016-02-20,Rosehill (AUS),4:05,First Seal (AUS),'B','',2015-04-11,Randwick (AUS),6:15,'',Dwelt - soon settled in midfield - 9th and pus...,2016-03-05,Randwick (AUS),3:50,'',
2,240574,2016-07-18,Chantilly (FR),2:20,Aladdine (GB),'A','',NaN,NaN,NaN,NULL,NaN,2016-08-15,Deauville (FR),1:50,'',
3,487574,2018-01-13,Gulfstream Park (USA),9:00,Ultra Brat (USA),'B','',2017-11-26,Aqueduct (USA),8:47,'',,2018-02-10,Gulfstream Park (USA),9:43,'',
4,719346,2019-05-25,San Isidro (ARG),9:35,Green Lemon (ARG),'V','',2018-05-25,San Isidro (ARG),11:00,'',,NaN,NaN,NaN,NULL,NaN
5,945843,2020-12-19,San Isidro (ARG),9:40,Cool Day (ARG),'B','',2020-10-31,San Isidro (ARG),9:20,'',,2021-10-30,San Isidro (ARG),8:45,'',
6,989658,2021-04-07,Compiegne (FR),11:10,Hacienda (FR),'A','',2021-03-01,Auteuil (FR),2:50,'',,2021-04-30,Auteuil (FR),4:05,'',


,letter_code,coded_runner_rows,current_hg_null_rows,current_hg_empty_rows,current_hg_populated_rows,rows_with_previous_run,rows_with_next_run,differs_from_previous_hg,differs_from_next_hg
0,A,2,0,2,0,1,2,0,0
1,B,4,0,4,0,4,4,0,0
2,V,1,0,1,0,1,0,0,0


### Equipment-change hypothesis result

The source does not support the suggestion that standalone `A`, `B` or `V` comments indicate a recorded headgear change.

Across all seven coded rows:

- the current `hg` value is empty in every case;
- no coded run has a populated equipment value;
- every available previous and next appearance also has an empty `hg` value;
- no coded run differs from the preceding or following `hg` state.

By code:

- `A`: 2 rows, no populated current or adjacent `hg`;
- `B`: 4 rows, no populated current or adjacent `hg`;
- `V`: 1 row, no populated current or adjacent `hg`.

The specialist suggestion was useful because it produced a direct falsifiable test. On the evidence available in this source, that hypothesis is **not supported**.

This does not prove that the letters cannot refer to equipment in the upstream publication, because the `hg` field may itself be incomplete or use jurisdiction-specific conventions. However, there is no positive source evidence linking these letters to headgear, and no stable correspondence can be inferred.

The correct treatment remains:

- preserve the raw letter unchanged;
- classify it as an `unresolved_source_code`;
- do not expand it into blinkers, visor or another equipment meaning;
- do not treat it as a standard Racing Post or Raceform convention without further evidence.

## 5. Profile embedded market and attributed-report markers

The field is not purely an in-running narrative. External research and the short-value evidence show that it can also contain:

- opening-price markers such as `op`;
- touched-price markers such as `tchd`;
- jockey, trainer or veterinary explanations;
- post-race findings or other attributed reports.

This stage will measure how often those structures occur and whether they appear:

- alone;
- appended to an ordinary close-up;
- in different capitalisation or punctuation forms;
- in particular periods, jurisdictions or courses;
- together within the same comment.

The analysis will remain marker-based rather than semantic. Detecting the exact text `op`, `tchd`, `jockey said`, `trainer said` or similar wording does not prove that every surrounding parenthetical has one uniform structure.

The purpose is to establish whether the field contains stable, recurring subfamilies that may justify later deterministic span extraction.

No parenthetical text will yet be removed from the narrative, parsed into canonical values or treated as independently verified fact.

In [5]:
# Profile recurring market and attributed-report markers inside populated comments.
#
# Matching is case-insensitive and deliberately literal. These flags identify
# candidate text families only; they do not yet parse spans or establish that
# every matching phrase has the same semantic meaning.
embedded_marker_profile = pd.read_sql_query(
    f"""
    WITH marked_comments AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            horse,
            comment,
            LOWER(comment) AS lower_comment,

            LOWER(comment) LIKE '%op %' AS has_opening_price_marker,
            LOWER(comment) LIKE '%tchd%' AS has_touched_price_marker,

            LOWER(comment) LIKE '%jockey said%' AS has_jockey_said,
            LOWER(comment) LIKE '%trainer said%' AS has_trainer_said,
            LOWER(comment) LIKE '%trainer''s representative%'
                AS has_trainer_representative,
            LOWER(comment) LIKE '%vet said%' AS has_vet_said,
            LOWER(comment) LIKE '%veterinary%' AS has_veterinary_wording,
            LOWER(comment) LIKE '%post-race examination%'
                AS has_post_race_examination
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND comment <> ''
    )
    SELECT
        COUNT(*) AS populated_rows,

        SUM(has_opening_price_marker) AS rows_with_opening_price_marker,
        SUM(has_touched_price_marker) AS rows_with_touched_price_marker,

        SUM(has_jockey_said) AS rows_with_jockey_said,
        SUM(has_trainer_said) AS rows_with_trainer_said,
        SUM(has_trainer_representative) AS rows_with_trainer_representative,
        SUM(has_vet_said) AS rows_with_vet_said,
        SUM(has_veterinary_wording) AS rows_with_veterinary_wording,
        SUM(has_post_race_examination) AS rows_with_post_race_examination,

        SUM(
            has_opening_price_marker
            OR has_touched_price_marker
        ) AS rows_with_any_market_marker,

        SUM(
            has_jockey_said
            OR has_trainer_said
            OR has_trainer_representative
            OR has_vet_said
            OR has_veterinary_wording
            OR has_post_race_examination
        ) AS rows_with_any_attributed_or_medical_marker,

        SUM(
            (
                has_opening_price_marker
                OR has_touched_price_marker
            )
            AND
            (
                has_jockey_said
                OR has_trainer_said
                OR has_trainer_representative
                OR has_vet_said
                OR has_veterinary_wording
                OR has_post_race_examination
            )
        ) AS rows_with_market_and_attributed_markers
    FROM marked_comments
    """,
    connection,
)

# Summarise the most common combinations of detected marker families.
#
# The combination label preserves only marker presence. It does not imply
# ordering, punctuation structure or independent factual validity.
embedded_marker_combinations = pd.read_sql_query(
    f"""
    WITH marked_comments AS (
        SELECT
            comment,

            LOWER(comment) LIKE '%op %' AS has_op,
            LOWER(comment) LIKE '%tchd%' AS has_tchd,
            LOWER(comment) LIKE '%jockey said%' AS has_jockey,
            (
                LOWER(comment) LIKE '%trainer said%'
                OR LOWER(comment) LIKE '%trainer''s representative%'
            ) AS has_trainer,
            (
                LOWER(comment) LIKE '%vet said%'
                OR LOWER(comment) LIKE '%veterinary%'
                OR LOWER(comment) LIKE '%post-race examination%'
            ) AS has_veterinary
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND comment <> ''
    )
    SELECT
        has_op,
        has_tchd,
        has_jockey,
        has_trainer,
        has_veterinary,
        COUNT(*) AS runner_rows
    FROM marked_comments
    WHERE
        has_op
        OR has_tchd
        OR has_jockey
        OR has_trainer
        OR has_veterinary
    GROUP BY
        has_op,
        has_tchd,
        has_jockey,
        has_trainer,
        has_veterinary
    ORDER BY runner_rows DESC
    """,
    connection,
)

# Retain a small lineage-bearing sample for each broad marker family.
#
# These examples are for structural inspection only. The complete raw comment
# remains intact and no parenthetical or attributed section is removed.
embedded_marker_examples = pd.read_sql_query(
    f"""
    WITH classified_comments AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            horse,
            comment,

            CASE
                WHEN LOWER(comment) LIKE '%jockey said%'
                    THEN 'jockey_said'
                WHEN (
                    LOWER(comment) LIKE '%trainer said%'
                    OR LOWER(comment) LIKE '%trainer''s representative%'
                )
                    THEN 'trainer_report'
                WHEN (
                    LOWER(comment) LIKE '%vet said%'
                    OR LOWER(comment) LIKE '%veterinary%'
                    OR LOWER(comment) LIKE '%post-race examination%'
                )
                    THEN 'veterinary_or_examination'
                WHEN LOWER(comment) LIKE '%tchd%'
                    THEN 'touched_price'
                WHEN LOWER(comment) LIKE '%op %'
                    THEN 'opening_price'
            END AS marker_family
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND comment <> ''
    ),
    ranked_examples AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY marker_family
                ORDER BY date, course, off, source_rowid
            ) AS example_number
        FROM classified_comments
        WHERE marker_family IS NOT NULL
    )
    SELECT
        marker_family,
        source_rowid,
        date,
        course,
        off,
        horse,
        LENGTH(comment) AS comment_length,
        comment
    FROM ranked_examples
    WHERE example_number <= 5
    ORDER BY marker_family, example_number
    """,
    connection,
)

display(embedded_marker_profile)
display(embedded_marker_combinations)
display(embedded_marker_examples)

,populated_rows,rows_with_opening_price_marker,rows_with_touched_price_marker,rows_with_jockey_said,rows_with_trainer_said,rows_with_trainer_representative,rows_with_vet_said,rows_with_veterinary_wording,rows_with_post_race_examination,rows_with_any_market_marker,rows_with_any_attributed_or_medical_marker,rows_with_market_and_attributed_markers
0,1510891,987473,517890,90584,9863,521,16429,170,2588,1153229,114221,99782


,has_op,has_tchd,has_jockey,has_trainer,has_veterinary,runner_rows
0,1,0,0,0,0,580827
1,1,1,0,0,0,321441
2,0,1,0,0,0,151179
3,1,0,1,0,0,41245
4,1,1,1,0,0,22864
5,0,0,1,0,0,11105
6,0,1,1,0,0,10991
7,1,0,0,0,1,6598
8,1,0,0,1,0,4439
9,1,1,0,0,1,3759


,marker_family,source_rowid,date,course,off,horse,comment_length,comment
0,jockey_said,414,2015-01-01,Catterick,12:30,Tickenwolf (IRE),116,In rear - behind when mistake 6th - tailed off...
1,jockey_said,420,2015-01-01,Catterick,1:05,Amazing Blue Sky (GB),110,Took keen hold - 2nd soon after 3rd - lost pla...
2,jockey_said,390,2015-01-01,Catterick,1:40,George Fernbeck (GB),114,Held up - pushed along 5th - driven 12th - wea...
3,jockey_said,447,2015-01-01,Catterick,3:25,Valsesia (IRE),170,Mid-division - prominent 5th - lost place 8th ...
4,jockey_said,430,2015-01-01,Cheltenham,12:45,King Of The Wolds (IRE),166,Held up - always in rear - mistake 4th - behin...
5,opening_price,2,2015-01-01,Catterick,12:30,Definitly Red (IRE),86,Tracked leaders - effort 3 out - led approachi...
6,opening_price,406,2015-01-01,Catterick,12:30,LAigle Royal (GER),98,Chased leaders - outpaced approaching 2 out - ...
7,opening_price,410,2015-01-01,Catterick,12:30,Palm Grey (IRE),92,Chased leaders - driven 3 out - outpaced appro...
8,opening_price,412,2015-01-01,Catterick,12:30,Tomorrows Legend (GB),65,Led - headed approaching 2 out - weakened betw...
9,opening_price,387,2015-01-01,Catterick,1:05,Mister Bishop (IRE),74,Chased leaders - outpaced 3 out - poor 7th whe...


### Embedded-marker findings

The `comment` field is not merely an in-running narrative. It routinely combines race description with betting-market history and, less frequently, attributed post-race explanations.

Across **1,510,891 populated comments**:

- **987,473**, or **65.36%**, contain an opening-price marker;
- **517,890**, or **34.28%**, contain a touched-price marker;
- **1,153,229**, or **76.33%**, contain at least one market marker;
- **114,221**, or **7.56%**, contain at least one attributed or medical marker;
- **99,782**, or **6.60%**, contain both market and attributed-report material.

Market information is therefore a normal component of the field rather than an occasional exception.

The most common detected structures are:

- opening-price marker without another detected family: **580,827 rows**;
- opening and touched-price markers together: **321,441 rows**;
- touched-price marker without an opening-price marker: **151,179 rows**.

This confirms that betting-history suffixes appear in several forms. An opening price is not always supplied, and a touched price may appear independently.

Attributed material is also substantial:

- `"jockey said"` appears in **90,584 rows**;
- `"trainer said"` appears in **9,863 rows**;
- `"trainer's representative"` appears in **521 rows**;
- `"vet said"` appears in **16,429 rows**;
- broader veterinary wording appears in **170 rows**;
- `"post-race examination"` appears in **2,588 rows**.

The examples show that these sections are normally appended to an ordinary close-up rather than stored separately. A single comment can therefore combine:

1. observed in-running events;
2. an attributed jockey or trainer explanation;
3. a veterinary or examination finding;
4. opening and touched betting prices.

This has several consequences.

First, the field mixes different evidential statuses:

- direct source observation;
- editorial interpretation;
- attributed explanation;
- veterinary or official report;
- market-history data.

These must not be treated as equivalent facts merely because they occur in one string.

Second, parentheses are not a single-purpose structure. They can contain:

- market movements;
- jockey explanations;
- trainer explanations;
- veterinary findings;
- combinations of several of these.

A rule that removes the final parenthetical or treats every parenthetical as betting data would therefore be unsafe.

Third, deterministic extraction may still be feasible for narrowly bounded spans introduced by explicit markers such as:

- `op`;
- `tchd`;
- `jockey said`;
- `trainer said`;
- `trainer's representative`;
- `vet said`;
- `post-race examination`.

However, marker detection alone is not yet sufficient. The next structural questions are:

- where each marked span begins and ends;
- whether marker spelling and punctuation vary by period;
- whether several parentheticals can occur in one comment;
- whether market spans are consistently terminal;
- whether attributed sections contain nested punctuation;
- whether any apparent matches are false positives.

Current conclusion:

> `comment` is a compound runner-level source field containing narrative close-up text, market-history annotations and attributed post-race material.

The raw comment must remain the authoritative source value. Any future extracted components must preserve their exact raw spans, marker type, position, extraction method and unresolved cases.

In [6]:
# Profile broad structural properties of comments containing recognised
# market or attributed-report markers.
#
# Parenthesis counts are measured from the raw stored text. Equal counts do
# not prove correct nesting, but unequal counts identify structurally unsafe
# cases immediately.
embedded_structure_profile = pd.read_sql_query(
    f"""
    WITH marked_comments AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            horse,
            comment,
            LOWER(comment) AS lower_comment,

            LENGTH(comment)
                - LENGTH(REPLACE(comment, '(', ''))
                AS opening_parenthesis_count,

            LENGTH(comment)
                - LENGTH(REPLACE(comment, ')', ''))
                AS closing_parenthesis_count,

            (
                LOWER(comment) LIKE '%op %'
                OR LOWER(comment) LIKE '%tchd%'
            ) AS has_market_marker,

            (
                LOWER(comment) LIKE '%jockey said%'
                OR LOWER(comment) LIKE '%trainer said%'
                OR LOWER(comment) LIKE '%trainer''s representative%'
                OR LOWER(comment) LIKE '%vet said%'
                OR LOWER(comment) LIKE '%veterinary%'
                OR LOWER(comment) LIKE '%post-race examination%'
            ) AS has_attributed_marker
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND comment <> ''
          AND (
              LOWER(comment) LIKE '%op %'
              OR LOWER(comment) LIKE '%tchd%'
              OR LOWER(comment) LIKE '%jockey said%'
              OR LOWER(comment) LIKE '%trainer said%'
              OR LOWER(comment) LIKE '%trainer''s representative%'
              OR LOWER(comment) LIKE '%vet said%'
              OR LOWER(comment) LIKE '%veterinary%'
              OR LOWER(comment) LIKE '%post-race examination%'
          )
    )
    SELECT
        COUNT(*) AS marked_rows,

        SUM(opening_parenthesis_count = closing_parenthesis_count)
            AS rows_with_equal_parenthesis_counts,

        SUM(opening_parenthesis_count <> closing_parenthesis_count)
            AS rows_with_unequal_parenthesis_counts,

        SUM(opening_parenthesis_count = 0)
            AS rows_without_parentheses,

        SUM(opening_parenthesis_count = 1)
            AS rows_with_one_parenthetical,

        SUM(opening_parenthesis_count > 1)
            AS rows_with_multiple_parentheticals,

        MAX(opening_parenthesis_count)
            AS maximum_opening_parentheses,

        SUM(SUBSTR(RTRIM(comment), -1) = ')')
            AS rows_ending_with_closing_parenthesis,

        SUM(
            has_market_marker
            AND SUBSTR(RTRIM(comment), -1) = ')'
        ) AS market_rows_ending_with_closing_parenthesis,

        SUM(
            has_attributed_marker
            AND SUBSTR(RTRIM(comment), -1) = ')'
        ) AS attributed_rows_ending_with_closing_parenthesis,

        SUM(
            has_market_marker
            AND has_attributed_marker
        ) AS rows_with_both_marker_families
    FROM marked_comments
    """,
    connection,
)

# Compare the broad structure of market-only, attributed-only and combined
# comments. This tests whether one simple suffix rule could plausibly apply
# across all three groups.
embedded_structure_by_family = pd.read_sql_query(
    f"""
    WITH marked_comments AS (
        SELECT
            comment,

            LENGTH(comment)
                - LENGTH(REPLACE(comment, '(', ''))
                AS opening_parenthesis_count,

            LENGTH(comment)
                - LENGTH(REPLACE(comment, ')', ''))
                AS closing_parenthesis_count,

            (
                LOWER(comment) LIKE '%op %'
                OR LOWER(comment) LIKE '%tchd%'
            ) AS has_market_marker,

            (
                LOWER(comment) LIKE '%jockey said%'
                OR LOWER(comment) LIKE '%trainer said%'
                OR LOWER(comment) LIKE '%trainer''s representative%'
                OR LOWER(comment) LIKE '%vet said%'
                OR LOWER(comment) LIKE '%veterinary%'
                OR LOWER(comment) LIKE '%post-race examination%'
            ) AS has_attributed_marker
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND comment <> ''
    )
    SELECT
        CASE
            WHEN has_market_marker AND has_attributed_marker
                THEN 'market_and_attributed'
            WHEN has_market_marker
                THEN 'market_only'
            WHEN has_attributed_marker
                THEN 'attributed_only'
        END AS marker_family,

        COUNT(*) AS runner_rows,

        SUM(opening_parenthesis_count = 0)
            AS rows_without_parentheses,

        SUM(opening_parenthesis_count = 1)
            AS rows_with_one_parenthetical,

        SUM(opening_parenthesis_count > 1)
            AS rows_with_multiple_parentheticals,

        SUM(opening_parenthesis_count <> closing_parenthesis_count)
            AS rows_with_unequal_parenthesis_counts,

        SUM(SUBSTR(RTRIM(comment), -1) = ')')
            AS rows_ending_with_closing_parenthesis
    FROM marked_comments
    WHERE has_market_marker OR has_attributed_marker
    GROUP BY marker_family
    ORDER BY marker_family
    """,
    connection,
)

# Inspect a small lineage-bearing sample of structurally difficult cases:
# unequal parenthesis counts, multiple parentheticals, and recognised markers
# in comments that do not end with a closing parenthesis.
#
# These cases will show whether simple terminal-parenthetical extraction would
# lose text or misclassify part of the narrative.
embedded_structure_exceptions = pd.read_sql_query(
    f"""
    WITH classified_comments AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            horse,
            comment,

            LENGTH(comment)
                - LENGTH(REPLACE(comment, '(', ''))
                AS opening_parenthesis_count,

            LENGTH(comment)
                - LENGTH(REPLACE(comment, ')', ''))
                AS closing_parenthesis_count,

            CASE
                WHEN (
                    LENGTH(comment)
                    - LENGTH(REPLACE(comment, '(', ''))
                ) <> (
                    LENGTH(comment)
                    - LENGTH(REPLACE(comment, ')', ''))
                )
                    THEN 'unequal_parenthesis_counts'

                WHEN (
                    LENGTH(comment)
                    - LENGTH(REPLACE(comment, '(', ''))
                ) > 1
                    THEN 'multiple_parentheticals'

                WHEN SUBSTR(RTRIM(comment), -1) <> ')'
                    THEN 'marker_not_terminal_parenthetical'
            END AS structural_case
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND comment <> ''
          AND (
              LOWER(comment) LIKE '%op %'
              OR LOWER(comment) LIKE '%tchd%'
              OR LOWER(comment) LIKE '%jockey said%'
              OR LOWER(comment) LIKE '%trainer said%'
              OR LOWER(comment) LIKE '%trainer''s representative%'
              OR LOWER(comment) LIKE '%vet said%'
              OR LOWER(comment) LIKE '%veterinary%'
              OR LOWER(comment) LIKE '%post-race examination%'
          )
    ),
    ranked_examples AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY structural_case
                ORDER BY date, course, off, source_rowid
            ) AS example_number
        FROM classified_comments
        WHERE structural_case IS NOT NULL
    )
    SELECT
        structural_case,
        source_rowid,
        date,
        course,
        off,
        horse,
        opening_parenthesis_count,
        closing_parenthesis_count,
        LENGTH(comment) AS comment_length,
        comment
    FROM ranked_examples
    WHERE example_number <= 10
    ORDER BY structural_case, example_number
    """,
    connection,
)

display(embedded_structure_profile)
display(embedded_structure_by_family)
display(embedded_structure_exceptions)

,marked_rows,rows_with_equal_parenthesis_counts,rows_with_unequal_parenthesis_counts,rows_without_parentheses,rows_with_one_parenthetical,rows_with_multiple_parentheticals,maximum_opening_parentheses,rows_ending_with_closing_parenthesis,market_rows_ending_with_closing_parenthesis,attributed_rows_ending_with_closing_parenthesis,rows_with_both_marker_families
0,1167668,1167661,7,419,1042339,124910,6,1167241,1152802,114221,99782


,marker_family,runner_rows,rows_without_parentheses,rows_with_one_parenthetical,rows_with_multiple_parentheticals,rows_with_unequal_parenthesis_counts,rows_ending_with_closing_parenthesis
0,attributed_only,14439,0,14266,173,0,14439
1,market_and_attributed,99782,0,159,99623,6,99782
2,market_only,1053447,419,1027914,25114,1,1053020


,structural_case,source_rowid,date,course,off,horse,opening_parenthesis_count,closing_parenthesis_count,comment_length,comment
0,marker_not_terminal_parenthetical,20478,2015-03-04,Kempton (AW),7:45,City Of Angkor Wat (IRE),0,0,162,Soon led and soon established 5 lengths lead -...
1,marker_not_terminal_parenthetical,21045,2015-03-06,Leicester,5:00,Share Option (GB),0,0,68,Set strong gallop - headed 8th - lost touch be...
2,marker_not_terminal_parenthetical,22153,2015-03-07,Gulfstream Park (USA),6:00,Ocean Telegraph (USA),0,0,137,Made all - set steady gallop and held narrow a...
3,marker_not_terminal_parenthetical,28038,2015-03-21,Rosehill (AUS),3:30,Contributer (IRE),0,0,218,Waited with in final trio - headway on inner o...
4,marker_not_terminal_parenthetical,28022,2015-03-21,Rosehill (AUS),4:50,Deep Field (AUS),0,0,115,Broke well from wide draw to lead and drop on ...
5,marker_not_terminal_parenthetical,30174,2015-03-28,Tampa Bay Downs (USA),9:53,Gavroche (USA),0,0,205,Waited with towards rear - took closer order b...
6,marker_not_terminal_parenthetical,33522,2015-04-04,Santa Anita (USA),11:30,Dortmund (USA),0,0,152,Made all - led on inner - gradually wound up t...
7,marker_not_terminal_parenthetical,33848,2015-04-05,Musselburgh,2:55,Valantino Oyster (IRE),0,0,82,Led at ordinary gallop - ridden and headed ove...
8,marker_not_terminal_parenthetical,35610,2015-04-08,Chantilly (FR),3:40,Spanish Romance (IRE),0,0,126,Prominent - front rank between horses from hal...
9,marker_not_terminal_parenthetical,36756,2015-04-11,Newcastle,3:55,Tapis Libre (GB),0,0,77,Led at modest gallop - ridden and headed over ...


In [7]:
# Re-display the structural exceptions without truncating the raw comment text.
#
# This changes notebook display settings only within this block. It does not
# alter the dataframe or any stored source values.
with pd.option_context(
    "display.max_colwidth",
    None,
    "display.max_rows",
    None,
    "display.width",
    240,
):
    display(
        embedded_structure_exceptions[
            [
                "structural_case",
                "source_rowid",
                "date",
                "course",
                "off",
                "horse",
                "opening_parenthesis_count",
                "closing_parenthesis_count",
                "comment",
            ]
        ]
    )

,structural_case,source_rowid,date,course,off,horse,opening_parenthesis_count,closing_parenthesis_count,comment
0,marker_not_terminal_parenthetical,20478,2015-03-04,Kempton (AW),7:45,City Of Angkor Wat (IRE),0,0,Soon led and soon established 5 lengths lead - came back to field 2f out - pushed along firmly and maintained gallop from over 1f out - never seriously challenged
1,marker_not_terminal_parenthetical,21045,2015-03-06,Leicester,5:00,Share Option (GB),0,0,Set strong gallop - headed 8th - lost touch before next - tailed off
2,marker_not_terminal_parenthetical,22153,2015-03-07,Gulfstream Park (USA),6:00,Ocean Telegraph (USA),0,0,Made all - set steady gallop and held narrow advantage - shaken up and quickened tempo from 2f out - ridden and edged clear final furlong
3,marker_not_terminal_parenthetical,28038,2015-03-21,Rosehill (AUS),3:30,Contributer (IRE),0,0,Waited with in final trio - headway on inner over 2 1/2f out - chased leader and edged left between horses 1 1/2f out - quickened to lead approaching final furlong and soon clear - well on top final 75yds - comfortably
4,marker_not_terminal_parenthetical,28022,2015-03-21,Rosehill (AUS),4:50,Deep Field (AUS),0,0,Broke well from wide draw to lead and drop on to rail - clear halfway - headed approaching final furlong - weakened
5,marker_not_terminal_parenthetical,30174,2015-03-28,Tampa Bay Downs (USA),9:53,Gavroche (USA),0,0,Waited with towards rear - took closer order before halfway - 4th and pushed along 2 1/2f out - quickened on outer to lead under 2f out - soon clear - driven inside final furlong - well on top final 100yds
6,marker_not_terminal_parenthetical,33522,2015-04-04,Santa Anita (USA),11:30,Dortmund (USA),0,0,Made all - led on inner - gradually wound up tempo from 3f out - 3 lengths clear and ridden over 1 1/2f out - pushed along and well on top final furlong
7,marker_not_terminal_parenthetical,33848,2015-04-05,Musselburgh,2:55,Valantino Oyster (IRE),0,0,Led at ordinary gallop - ridden and headed over 2f out - weakened well over 1f out
8,marker_not_terminal_parenthetical,35610,2015-04-08,Chantilly (FR),3:40,Spanish Romance (IRE),0,0,Prominent - front rank between horses from halfway - led 2f out - driven clear inside final furlong - well on top final 100yds
9,marker_not_terminal_parenthetical,36756,2015-04-11,Newcastle,3:55,Tapis Libre (GB),0,0,Led at modest gallop - ridden and headed over 2f out - weakened final furlong


### Structural inspection and marker-detection correction

The full exception text shows that the initial opening-price detector was over-broad.

The condition `LOWER(comment) LIKE '%op %'` does not identify only the betting abbreviation `op`. It also matches the same character sequence inside ordinary narrative text, including:

- `gallop -`
- `drop on to rail`
- `well on top final 100yds`

The **419 detected marker rows without parentheses** therefore cannot be interpreted as market annotations outside parenthetical structures. At least the inspected examples are false-positive substring matches produced by the exploratory query.

This is a useful methodological finding:

> A familiar short marker cannot be detected safely without testing its lexical boundaries against the full source vocabulary.

The earlier headline count of **987,473 opening-price-marker rows** may consequently be slightly overstated. The market-family counts and structural summaries must be recalculated using more precise source-presented forms, including candidate patterns such as:

- `(op `
- `(Op. `
- ` tchd `
- `(tchd `

The attributed-report findings are not invalidated by this specific problem because phrases such as `jockey said`, `trainer said` and `vet said` are substantially more specific. Their punctuation structure still requires investigation, but they are less exposed to accidental matches inside ordinary words.

The full structural examples also establish several genuine properties:

- attributed explanations and market history commonly occupy separate consecutive parentheticals;
- comments can contain nested descriptive parentheses inside an attributed report;
- seven rows have unequal parenthesis counts;
- those seven include missing, duplicated or malformed closing parentheses rather than one stable alternative syntax;
- a simple rule based only on the final opening parenthesis would be unsafe;
- market annotations generally appear terminally in the inspected genuine examples;
- raw malformed punctuation must be preserved rather than repaired silently.

No structural extraction rule is authorised from the exploratory counts. The market-marker population must first be reprofiled with corrected boundaries.

In [8]:
# Reprofile market markers using source-presented boundary forms rather than
# the over-broad substring `%op %`, which also matched ordinary words such as
# `gallop` and phrases such as `drop on`.
#
# These patterns remain descriptive candidates. They are narrower than the
# first pass but do not yet constitute a production parser.
corrected_market_marker_profile = pd.read_sql_query(
    f"""
    WITH marked_comments AS (
        SELECT
            comment,
            LOWER(comment) AS lower_comment,

            (
                LOWER(comment) LIKE '%(op %'
                OR LOWER(comment) LIKE '%(op. %'
            ) AS has_opening_price_marker,

            (
                LOWER(comment) LIKE '% tchd %'
                OR LOWER(comment) LIKE '%(tchd %'
                OR LOWER(comment) LIKE '% tchd %)%'
                OR LOWER(comment) LIKE '% tchd % and tchd %'
            ) AS has_touched_price_marker,

            (
                LOWER(comment) LIKE '%jockey said%'
                OR LOWER(comment) LIKE '%trainer said%'
                OR LOWER(comment) LIKE '%trainer''s representative%'
                OR LOWER(comment) LIKE '%vet said%'
                OR LOWER(comment) LIKE '%veterinary%'
                OR LOWER(comment) LIKE '%post-race examination%'
            ) AS has_attributed_marker,

            LENGTH(comment)
                - LENGTH(REPLACE(comment, '(', ''))
                AS opening_parenthesis_count,

            LENGTH(comment)
                - LENGTH(REPLACE(comment, ')', ''))
                AS closing_parenthesis_count
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND comment <> ''
    )
    SELECT
        COUNT(*) AS populated_rows,

        SUM(has_opening_price_marker)
            AS rows_with_opening_price_marker,

        SUM(has_touched_price_marker)
            AS rows_with_touched_price_marker,

        SUM(
            has_opening_price_marker
            OR has_touched_price_marker
        ) AS rows_with_any_market_marker,

        SUM(has_attributed_marker)
            AS rows_with_any_attributed_marker,

        SUM(
            has_attributed_marker
            AND (
                has_opening_price_marker
                OR has_touched_price_marker
            )
        ) AS rows_with_market_and_attributed_markers,

        SUM(
            (
                has_opening_price_marker
                OR has_touched_price_marker
            )
            AND opening_parenthesis_count = 0
        ) AS market_rows_without_parentheses,

        SUM(
            (
                has_opening_price_marker
                OR has_touched_price_marker
            )
            AND SUBSTR(RTRIM(comment), -1) = ')'
        ) AS market_rows_ending_with_closing_parenthesis,

        SUM(
            (
                has_opening_price_marker
                OR has_touched_price_marker
                OR has_attributed_marker
            )
            AND opening_parenthesis_count <> closing_parenthesis_count
        ) AS marked_rows_with_unequal_parenthesis_counts
    FROM marked_comments
    """,
    connection,
)

# Compare the corrected marker combinations.
#
# This shows how often opening-price, touched-price and attributed material
# coexist after removing the known false-positive `op` matches.
corrected_market_marker_combinations = pd.read_sql_query(
    f"""
    WITH marked_comments AS (
        SELECT
            (
                LOWER(comment) LIKE '%(op %'
                OR LOWER(comment) LIKE '%(op. %'
            ) AS has_op,

            (
                LOWER(comment) LIKE '% tchd %'
                OR LOWER(comment) LIKE '%(tchd %'
                OR LOWER(comment) LIKE '% tchd %)%'
                OR LOWER(comment) LIKE '% tchd % and tchd %'
            ) AS has_tchd,

            (
                LOWER(comment) LIKE '%jockey said%'
                OR LOWER(comment) LIKE '%trainer said%'
                OR LOWER(comment) LIKE '%trainer''s representative%'
                OR LOWER(comment) LIKE '%vet said%'
                OR LOWER(comment) LIKE '%veterinary%'
                OR LOWER(comment) LIKE '%post-race examination%'
            ) AS has_attributed
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND comment <> ''
    )
    SELECT
        has_op,
        has_tchd,
        has_attributed,
        COUNT(*) AS runner_rows
    FROM marked_comments
    WHERE has_op OR has_tchd OR has_attributed
    GROUP BY
        has_op,
        has_tchd,
        has_attributed
    ORDER BY runner_rows DESC
    """,
    connection,
)

# Inspect any remaining market-marker rows without parentheses.
#
# A genuine market marker would normally be expected inside a parenthetical.
# Any residue here may reveal further false positives or an alternative format.
corrected_market_marker_exceptions = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        horse,
        comment
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND comment <> ''
      AND (
          LOWER(comment) LIKE '%(op %'
          OR LOWER(comment) LIKE '%(op. %'
          OR LOWER(comment) LIKE '% tchd %'
          OR LOWER(comment) LIKE '%(tchd %'
          OR LOWER(comment) LIKE '% tchd %)%'
          OR LOWER(comment) LIKE '% tchd % and tchd %'
      )
      AND (
          LENGTH(comment)
          - LENGTH(REPLACE(comment, '(', ''))
      ) = 0
    ORDER BY date, course, off, rowid
    LIMIT 50
    """,
    connection,
)

display(corrected_market_marker_profile)
display(corrected_market_marker_combinations)

with pd.option_context(
    "display.max_colwidth",
    None,
    "display.max_rows",
    None,
    "display.width",
    240,
):
    display(corrected_market_marker_exceptions)

,populated_rows,rows_with_opening_price_marker,rows_with_touched_price_marker,rows_with_any_market_marker,rows_with_any_attributed_marker,rows_with_market_and_attributed_markers,market_rows_without_parentheses,market_rows_ending_with_closing_parenthesis,marked_rows_with_unequal_parenthesis_counts
0,1510891,979671,517771,1150592,114221,99461,0,1150592,7


,has_op,has_tchd,has_attributed,runner_rows
0,1,0,0,578624
1,1,1,0,316753
2,0,1,0,155754
3,1,0,1,54197
4,1,1,1,30097
5,0,1,1,15167
6,0,0,1,14760


,source_rowid,date,course,off,horse,comment


### Corrected market-marker findings

The refined boundary patterns remove the false positives caused by matching `op` inside ordinary words.

Across **1,510,891 populated comments**:

- **979,671** contain a recognised opening-price form;
- **517,771** contain a recognised touched-price form;
- **1,150,592** contain at least one recognised market marker;
- **114,221** contain at least one attributed-report marker;
- **99,461** contain both market and attributed material.

The corrected opening-price count is **7,802 rows lower** than the exploratory count. This confirms that the original `%op %` condition materially overstated the population by matching ordinary narrative text.

The touched-price count falls by only **119 rows**, indicating that `tchd` was already a comparatively distinctive marker.

The corrected marker combinations are:

- opening price only: **578,624 rows**;
- opening and touched prices: **316,753 rows**;
- touched price without opening price: **155,754 rows**;
- opening price with attributed material: **54,197 rows**;
- opening and touched prices with attributed material: **30,097 rows**;
- touched price with attributed material: **15,167 rows**;
- attributed material without a detected market marker: **14,760 rows**.

All **1,150,592** corrected market-marker rows:

- contain at least one opening parenthesis;
- end with a closing parenthesis.

The exception query returned no rows. This strongly supports the proposition that recognised market-history material is presented inside a terminal parenthetical structure.

However, terminal placement alone does not justify extracting the final parenthetical blindly. Comments can contain:

- a separate attributed parenthetical followed by a market parenthetical;
- nested descriptive parentheses within an attributed report;
- malformed or unbalanced punctuation.

Seven marked comments have unequal opening and closing parenthesis counts. These remain explicit structural exceptions and must not be silently repaired.

Current conclusion:

> Opening-price and touched-price annotations form a large, stable and normally terminal component of the `comment` field, but safe extraction requires marker-aware parsing rather than generic parenthesis removal.

The corrected counts supersede the earlier exploratory market-marker counts.

## 7. Investigate exceptionally long comments

The longest populated comments reach more than 2,000 characters, far beyond the field’s mean length of approximately 122 characters.

Visible beginnings of the longest examples resemble ordinary in-running close-ups, but their exceptional size could result from:

- unusually detailed attributed explanations;
- multiple appended reports;
- repeated or duplicated text;
- accidental concatenation of several runner comments;
- malformed punctuation;
- another source or extraction artefact.

This stage will inspect the long-comment population without assuming corruption.

The investigation will establish:

- how many comments exceed selected high-length thresholds;
- their distribution by period, course and jurisdiction;
- whether they are concentrated in particular races or runners;
- counts of market, attributed and veterinary markers;
- parenthesis balance and structural complexity;
- repeated text fragments within individual comments;
- whether the longest values contain apparent material belonging to other runners.

Small lineage-bearing samples will be inspected in full. No long comment will be truncated, repaired or excluded solely because of its length.

In [9]:
# Profile the population of exceptionally long comments.
#
# The thresholds are descriptive only. Length by itself does not establish
# corruption, duplication or a different source type.
long_comment_profile = pd.read_sql_query(
    f"""
    WITH long_comments AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            horse,
            comment,
            LENGTH(comment) AS comment_length,

            LENGTH(comment)
                - LENGTH(REPLACE(comment, '(', ''))
                AS opening_parenthesis_count,

            LENGTH(comment)
                - LENGTH(REPLACE(comment, ')', ''))
                AS closing_parenthesis_count,

            (
                LOWER(comment) LIKE '%jockey said%'
                OR LOWER(comment) LIKE '%trainer said%'
                OR LOWER(comment) LIKE '%trainer''s representative%'
                OR LOWER(comment) LIKE '%vet said%'
                OR LOWER(comment) LIKE '%veterinary%'
                OR LOWER(comment) LIKE '%post-race examination%'
            ) AS has_attributed_marker,

            (
                LOWER(comment) LIKE '%(op %'
                OR LOWER(comment) LIKE '%(op. %'
                OR LOWER(comment) LIKE '% tchd %'
                OR LOWER(comment) LIKE '%(tchd %'
            ) AS has_market_marker
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND comment <> ''
          AND LENGTH(comment) > 500
    )
    SELECT
        COUNT(*) AS rows_over_500_characters,
        SUM(comment_length > 750) AS rows_over_750_characters,
        SUM(comment_length > 1000) AS rows_over_1000_characters,
        SUM(comment_length > 1500) AS rows_over_1500_characters,
        SUM(comment_length > 2000) AS rows_over_2000_characters,

        MIN(comment_length) AS minimum_long_comment_length,
        MAX(comment_length) AS maximum_long_comment_length,
        ROUND(AVG(comment_length), 2) AS mean_long_comment_length,

        COUNT(DISTINCT course) AS distinct_courses,
        COUNT(
            DISTINCT date || '|' || course || '|' || off
        ) AS provisional_races,

        SUM(has_attributed_marker) AS rows_with_attributed_marker,
        SUM(has_market_marker) AS rows_with_market_marker,

        SUM(
            opening_parenthesis_count
            <> closing_parenthesis_count
        ) AS rows_with_unequal_parenthesis_counts,

        MAX(opening_parenthesis_count)
            AS maximum_opening_parentheses
    FROM long_comments
    """,
    connection,
)

# Summarise where long comments are concentrated.
#
# This helps distinguish a general field property from a course-, period- or
# feed-specific behaviour.
long_comment_course_profile = pd.read_sql_query(
    f"""
    SELECT
        course,
        COUNT(*) AS runner_rows,
        COUNT(
            DISTINCT date || '|' || course || '|' || off
        ) AS provisional_races,
        MIN(date) AS first_date,
        MAX(date) AS last_date,
        MIN(LENGTH(comment)) AS minimum_length,
        MAX(LENGTH(comment)) AS maximum_length,
        ROUND(AVG(LENGTH(comment)), 2) AS mean_length
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND comment <> ''
      AND LENGTH(comment) > 500
    GROUP BY course
    ORDER BY runner_rows DESC, course
    LIMIT 30
    """,
    connection,
)

# Inspect the longest comments in full with source lineage and basic
# structural indicators.
#
# The raw text remains unchanged. Full display is needed to determine whether
# these values contain repeated clauses, appended reports or concatenation.
long_comment_examples = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        horse,
        LENGTH(comment) AS comment_length,

        LENGTH(comment)
            - LENGTH(REPLACE(comment, '(', ''))
            AS opening_parenthesis_count,

        LENGTH(comment)
            - LENGTH(REPLACE(comment, ')', ''))
            AS closing_parenthesis_count,

        comment
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND comment <> ''
      AND LENGTH(comment) > 500
    ORDER BY comment_length DESC, rowid
    LIMIT 20
    """,
    connection,
)

display(long_comment_profile)
display(long_comment_course_profile)

with pd.option_context(
    "display.max_colwidth",
    None,
    "display.max_rows",
    None,
    "display.width",
    260,
):
    display(long_comment_examples)

,rows_over_500_characters,rows_over_750_characters,rows_over_1000_characters,rows_over_1500_characters,rows_over_2000_characters,minimum_long_comment_length,maximum_long_comment_length,mean_long_comment_length,distinct_courses,provisional_races,rows_with_attributed_marker,rows_with_market_marker,rows_with_unequal_parenthesis_counts,maximum_opening_parentheses
0,1190,538,265,47,4,501,2206,815.84,103,1170,895,1022,1,4


,course,runner_rows,provisional_races,first_date,last_date,minimum_length,maximum_length,mean_length
0,Wolverhampton (AW),93,92,2015-01-26,2026-03-17,515,2206,1112.61
1,Lingfield (AW),44,43,2015-01-07,2026-02-27,501,1571,781.89
2,Southwell (AW),39,35,2016-01-26,2026-02-28,506,1303,845.08
3,Dundalk (AW) (IRE),35,34,2015-01-30,2024-02-23,509,1417,718.11
4,Navan (IRE),32,31,2015-03-28,2025-03-22,518,1548,798.28
5,Ludlow,30,30,2015-01-15,2026-02-18,506,1742,924.67
6,Fairyhouse (IRE),29,28,2015-11-28,2025-04-20,505,1372,727.59
7,Uttoxeter,29,29,2015-02-07,2026-05-02,523,1424,936.90
8,Limerick (IRE),24,24,2015-11-08,2025-07-12,501,1373,814.63
9,Kempton (AW),23,23,2015-01-28,2026-01-28,506,1413,828.30


,source_rowid,date,course,off,horse,comment_length,opening_parenthesis_count,closing_parenthesis_count,comment
0,1622460,2025-01-13,Wolverhampton (AW),5:30,Further Measure (USA),2206,2,2,Raced in last - still plenty to do when switched right over 1f out - ran on but never near to challenge (inquiry held into running and riding of gelding which raced in rear throughout before staying on strongly downhome straight from an unpromising position to finish 4th of 7 runners - beaten 2 1/4 lengths; jockey stated his instructions were to jump out - sit in box seat and see how race developed from there; he explained that gelding had missed the kick and he therefore accepted his early position and decided to sit last; he indicated that he thought the pace was decent for first 2f before it steadied up leaving back straight on first circuit leaving him further out of his ground than had been intended; from this point in race and during middle stages jockey acknowledged that he was aware pace was slow but he was reluctant to make a forward move to improve his position for fear of getting trapped wide; in explaining his riding from leaving back straight on final circuit - he stated that he was again hesitant to switch wide - due to concerns he could end up at least 4 wide around outside of runners - so committed to ride for luck down inner in hope of obtaining a clear passage between rivals on rail or to the inside of a rival if runners fanned approaching home bend - which they often do at Wolverhampton - but it failed to materialise on this occasion and he was denied clear running for some distance approaching2f marker; on straightening for home he managed to switch gelding to outside - mounting his challenge and gelding had stayed on very well through to line; trainer confirmed instructions; however expressed disappointment that even allowing for fact that gelding wasn't quickest into stride - greater effort had not been made by rider to ensure gelding settled more prominently and was ridden to instructions; trainer also indicated that he was dissatisfied with fact that jockey failed to switch wider and improve his position on to the back of favourite for some distance from approaching 3f marker until entering home straight; jockey told stewards he was still very inexperienced riding in staying races and only had approximately 50 career rides to his name)(op 15/8)
1,1745480,2025-10-06,Wolverhampton (AW),4:20,Oman (IRE),2083,2,2,In rear and raced wide early - still plenty to do and waiting for room over 1f out - nudged along and some headway inside final furlong (enq held into the running and riding of the gelding - which jumped well from stall 11 and was restrained to settle at the rear of the field - before appearing to be tenderly handled inside the final 3f to finish ninth of thirteen runners - beaten by 5 1/4l. Vet examined the gelding on two occasions post-race and had nothing to report. Jockey stated that her instructions were to be positive - obtain a position around mid-division and hopefully the gelding would be staying on at the finish. Jockey explained that from her wide draw - she found herself further back than intended off what she considered only a steady tempo. Jockey acknowledged that passing the 3f marker despite being poorly positioned - she had been slow to commit with her finishing effort - but indicated that off the home bend once she had chosen to mount her challenge down the centre of the track - she encountered traffic problems and was denied clear running both approaching and inside the final furlong - which had prevented her from being able to ride more vigorously. Trainer confirmed the instructions and conceded that the gelding had been further out of his ground than he had anticipated - before agreeing that the rider had been slow to ask the gelding to improve his position between 3f and 2f out - however he felt his rider had been a victim of circumstances in the home straight and did not have much luck in runn

### Exceptionally long-comment findings

The exceptionally long comments are genuine extended source reports rather than a distinct unrelated text type.

Across the populated source:

- **1,190 comments** exceed 500 characters;
- **538** exceed 750 characters;
- **265** exceed 1,000 characters;
- **47** exceed 1,500 characters;
- **4** exceed 2,000 characters;
- the longest contains **2,206 characters**.

These rows are rare, representing approximately **0.079%** of populated comments.

They are distributed across:

- **103 courses**;
- **1,170 provisional races**;
- the full study period rather than one isolated import batch.

The population is concentrated most visibly at British and Irish courses, particularly Wolverhampton, Lingfield, Southwell, Dundalk, Navan, Ludlow and other tracks where detailed steward inquiries and post-race explanations are commonly recorded.

The longest comments consistently contain:

1. a normal in-running close-up;
2. an extended inquiry, steward, jockey, trainer or veterinary account;
3. sometimes a market-history parenthetical at the end.

Of the 1,190 long comments:

- **895** contain an attributed-report marker;
- **1,022** contain a recognised market marker;
- only **1** has unequal parenthesis counts;
- up to **4** opening parentheses occur in one comment.

The full examples show no evidence that the longest values are random corruption or concatenation of several unrelated runner comments. Instead, they contain coherent runner-specific inquiry narratives, including:

- riding instructions;
- explanations from jockeys and trainers;
- veterinary findings;
- steward decisions;
- suspensions, bans or fines;
- betting-market observations;
- discussion of race pace, position and interference.

The extreme length is therefore primarily explained by embedded official or attributed reports rather than unusually verbose close-up prose.

Several examples also demonstrate that the field can contain information of materially different evidential status inside one raw value:

- race-reader observation;
- rider or trainer testimony;
- veterinary examination;
- steward interpretation;
- regulatory outcome;
- betting-market history.

This confirms that the field must not be treated as one homogeneous narrative.

The long comments also expose minor source-quality issues, including:

- inconsistent capitalisation;
- missing spaces;
- spelling and transcription errors;
- punctuation inconsistencies;
- unbalanced parentheses in a very small residue;
- changing terminology such as `enquiry`, `inquiry` and `enq`.

These imperfections do not justify rewriting the source text.

Current conclusion:

> Exceptionally long comments are valid compound runner records dominated by steward inquiries and attributed post-race evidence, not a separate corrupted text population.

No length-based exclusion or truncation is justified. Any future extraction should separate clearly marked subcomponents while retaining the complete raw comment and source lineage.

## 8. Profile comment coverage across the source

The field’s physical behaviour and internal structures are now clearer, but analytical use also depends on where comments are present and where they are absent.

This stage will measure comment coverage by:

- calendar year;
- course and jurisdiction label;
- race type;
- provisional race;
- runner position within the source race;
- complete versus partial race coverage.

The investigation will distinguish:

- races where every stored runner has substantive text;
- races where some runners have text and others are empty;
- races where all runner comments are empty;
- races containing probable placeholders or unresolved source codes;
- changes in coverage across periods and source regions.

This matters because a high overall populated-row rate can conceal selective coverage. For example, comments may be concentrated among leading finishers, particular jurisdictions, domestic racing, recent years or races covered by a particular editorial feed.

The stage will describe source availability only. It will not assume that a populated comment is equally detailed, accurate or analytically useful in every jurisdiction or period.

In [10]:
# Measure comment coverage at runner and provisional-race level by calendar year.
#
# A provisional race is identified by date, course and off time, matching the
# current notebook grain. This does not replace the project’s governed race
# identity work; it is used here only to describe comment availability.
#
# Source values are grouped into:
# - empty string;
# - probable placeholder or unresolved short code;
# - other populated text.
#
# The classification is analytical only. Raw comment values remain unchanged.
comment_coverage_by_year = pd.read_sql_query(
    f"""
    WITH classified_rows AS (
        SELECT
            SUBSTR(date, 1, 4) AS calendar_year,
            date,
            course,
            off,
            comment,

            CASE
                WHEN comment = ''
                    THEN 'empty_string'

                WHEN comment IN (
                    '.',
                    '..',
                    '-',
                    ' -',
                    '/',
                    'A',
                    'B',
                    'V',
                    '1'
                )
                    THEN 'placeholder_or_unresolved_code'

                ELSE 'other_populated_text'
            END AS comment_state
        FROM data
        WHERE {DATA_ROW_PREDICATE}
    ),
    race_coverage AS (
        SELECT
            calendar_year,
            date,
            course,
            off,

            COUNT(*) AS stored_runner_rows,

            SUM(comment_state = 'empty_string')
                AS empty_runner_rows,

            SUM(
                comment_state = 'placeholder_or_unresolved_code'
            ) AS placeholder_or_code_rows,

            SUM(
                comment_state = 'other_populated_text'
            ) AS other_populated_rows
        FROM classified_rows
        GROUP BY
            calendar_year,
            date,
            course,
            off
    )
    SELECT
        calendar_year,

        SUM(stored_runner_rows) AS stored_runner_rows,
        COUNT(*) AS provisional_races,

        SUM(empty_runner_rows) AS empty_runner_rows,
        SUM(placeholder_or_code_rows)
            AS placeholder_or_code_rows,
        SUM(other_populated_rows) AS other_populated_rows,

        ROUND(
            100.0 * SUM(other_populated_rows)
            / SUM(stored_runner_rows),
            2
        ) AS other_populated_runner_percentage,

        SUM(empty_runner_rows = stored_runner_rows)
            AS races_all_empty,

        SUM(
            other_populated_rows = stored_runner_rows
        ) AS races_all_other_populated,

        SUM(
            empty_runner_rows > 0
            AND other_populated_rows > 0
        ) AS races_mixed_empty_and_populated,

        SUM(placeholder_or_code_rows > 0)
            AS races_with_placeholder_or_code,

        SUM(
            other_populated_rows = 0
            AND placeholder_or_code_rows > 0
        ) AS races_with_codes_but_no_other_text
    FROM race_coverage
    GROUP BY calendar_year
    ORDER BY calendar_year
    """,
    connection,
)

# Summarise complete, partial and absent race coverage across the full source.
#
# Placeholder and unresolved-code rows are kept separate from ordinary
# populated commentary so that nominally non-empty text does not inflate the
# substantive coverage estimate.
overall_race_comment_coverage = pd.read_sql_query(
    f"""
    WITH classified_rows AS (
        SELECT
            date,
            course,
            off,

            CASE
                WHEN comment = ''
                    THEN 'empty_string'

                WHEN comment IN (
                    '.',
                    '..',
                    '-',
                    ' -',
                    '/',
                    'A',
                    'B',
                    'V',
                    '1'
                )
                    THEN 'placeholder_or_unresolved_code'

                ELSE 'other_populated_text'
            END AS comment_state
        FROM data
        WHERE {DATA_ROW_PREDICATE}
    ),
    race_coverage AS (
        SELECT
            date,
            course,
            off,

            COUNT(*) AS stored_runner_rows,

            SUM(comment_state = 'empty_string')
                AS empty_runner_rows,

            SUM(
                comment_state = 'placeholder_or_unresolved_code'
            ) AS placeholder_or_code_rows,

            SUM(
                comment_state = 'other_populated_text'
            ) AS other_populated_rows
        FROM classified_rows
        GROUP BY date, course, off
    )
    SELECT
        COUNT(*) AS provisional_races,
        SUM(stored_runner_rows) AS stored_runner_rows,

        SUM(empty_runner_rows = stored_runner_rows)
            AS races_all_empty,

        SUM(other_populated_rows = stored_runner_rows)
            AS races_all_other_populated,

        SUM(
            empty_runner_rows > 0
            AND other_populated_rows > 0
        ) AS races_mixed_empty_and_populated,

        SUM(placeholder_or_code_rows > 0)
            AS races_with_placeholder_or_code,

        SUM(
            other_populated_rows = 0
            AND placeholder_or_code_rows > 0
        ) AS races_with_codes_but_no_other_text,

        SUM(empty_runner_rows) AS empty_runner_rows,
        SUM(placeholder_or_code_rows)
            AS placeholder_or_code_rows,
        SUM(other_populated_rows) AS other_populated_rows,

        ROUND(
            100.0 * SUM(other_populated_rows)
            / SUM(stored_runner_rows),
            2
        ) AS other_populated_runner_percentage
    FROM race_coverage
    """,
    connection,
)

display(overall_race_comment_coverage)
display(comment_coverage_by_year)

,provisional_races,stored_runner_rows,races_all_empty,races_all_other_populated,races_mixed_empty_and_populated,races_with_placeholder_or_code,races_with_codes_but_no_other_text,empty_runner_rows,placeholder_or_code_rows,other_populated_rows,other_populated_runner_percentage
0,189043,1851285,21560,156148,11305,58,20,340394,238,1510653,81.6


,calendar_year,stored_runner_rows,provisional_races,empty_runner_rows,placeholder_or_code_rows,other_populated_rows,other_populated_runner_percentage,races_all_empty,races_all_other_populated,races_mixed_empty_and_populated,races_with_placeholder_or_code,races_with_codes_but_no_other_text
0,2015,157683,16609,34676,3,123004,78.01,2093,13182,1334,3,0
1,2016,157938,16235,30306,2,127630,80.81,1757,13338,1139,2,1
2,2017,167494,17110,29140,1,138353,82.60,1714,14301,1095,1,0
3,2018,172047,17546,31168,23,140856,81.87,1762,14470,1310,6,1
4,2019,173039,17345,32223,3,140813,81.38,1860,14175,1309,3,1
5,2020,121683,11875,23653,75,97955,80.50,1622,9507,734,12,12
6,2021,175552,17706,29595,16,145941,83.13,1800,14798,1107,4,1
7,2022,167479,17350,30328,0,137151,81.89,2049,14413,888,0,0
8,2023,156075,16119,24738,15,131322,84.14,1689,13738,690,3,0
9,2024,169702,17170,30677,0,139025,81.92,2112,14289,769,0,0


### Overall and temporal coverage findings

The `comment` field has substantial but incomplete source coverage.

Across **1,851,285 stored runner rows** and **189,043 provisional races**:

- **1,510,653 runner rows**, or **81.60%**, contain ordinary populated text;
- **340,394 rows**, or **18.39%**, contain an empty string;
- **238 rows**, or approximately **0.013%**, contain a probable placeholder or unresolved short code.

At provisional-race level:

- **156,148 races** have ordinary populated text for every stored runner;
- **21,560 races** have empty comments for every stored runner;
- **11,305 races** contain a mixture of empty and ordinary populated comments;
- **58 races** contain at least one probable placeholder or unresolved code;
- **20 races** contain placeholder or coded values but no ordinary populated comment.

Approximately:

- **82.6%** of provisional races have ordinary comments for every stored runner;
- **11.4%** have no supplied comment text for any stored runner;
- **6.0%** contain mixed empty and populated coverage.

These broad race categories do not form a perfectly exhaustive partition because races containing placeholder or unresolved-code values can fall outside the simple all-empty, all-ordinary-populated and mixed-empty-and-populated definitions.

### Temporal behaviour

Ordinary populated runner coverage remains broadly stable throughout the source period, generally between **78% and 84%**.

Observed yearly coverage includes:

- **2015:** 78.01%;
- **2016:** 80.81%;
- **2017:** 82.60%;
- **2018:** 81.87%;
- **2019:** 81.38%;
- **2020:** 80.50%;
- **2021:** 83.13%;
- **2022:** 81.89%;
- **2023:** 84.14%;
- **2024:** 81.92%;
- **2025:** 81.28%;
- **2026 to the source endpoint:** 80.58%.

There is no simple progression from poor historic coverage to complete modern coverage. Missing comments remain a persistent source feature.

The lowest observed yearly ordinary-text coverage is **78.01% in 2015**, while the highest is **84.14% in 2023**. The relatively narrow range suggests that overall comment availability is structurally stable, although the jurisdictions and race types contributing to missingness may change.

Placeholder and unresolved-code values remain negligible in every year. Their largest visible concentrations are:

- **75 rows in 2020**;
- **62 rows in the partial 2026 population**;
- **38 rows in 2025**;
- **23 rows in 2018**.

This concentration is driven largely by a small number of affected races rather than a widespread field convention.

### Current conclusion

The field is well populated overall, but it is not complete.

A populated-runner rate of **81.60%** is insufficient evidence that comments are available uniformly. More than **21,000 races** have no comment text for any stored runner, while over **11,000 races** have selective runner-level coverage.

The next coverage question is therefore not whether comments are generally common, but **where the missing and partial race coverage occurs**.

Course, jurisdiction and race-type profiling is required before the field can be used for comparative analysis without selection bias.

## 9. Locate missing and partial comment coverage

Overall coverage is stable but incomplete. The next step is to identify where empty and selectively populated comments are concentrated.

This stage will profile coverage by:

- course;
- inferred jurisdiction from the course label;
- race type;
- complete, partial and absent race-level coverage;
- concentration of placeholder and unresolved-code values.

The purpose is to distinguish broad source behaviour from local feed behaviour.

In particular, the analysis will test whether missingness is associated with:

- overseas jurisdictions;
- particular courses;
- Flat, jump or other race types;
- whole races lacking commentary;
- selective runner-level commentary within otherwise populated races.

Course and jurisdiction labels will be treated as source-presented classifications. Any jurisdiction inference used here is descriptive and will not replace the project’s governed course-identity reference.

No absent comment will be imputed, and no course or race-type comparison will assume that populated comments are equivalent in depth or quality.

In [11]:
# Profile comment coverage by source-presented course label and race type.
#
# This step deliberately avoids inventing a new jurisdiction mapping from the
# course string. Jurisdiction coverage will be joined to the project’s governed
# course reference after the source-label concentrations are understood.
#
# Probable placeholders and unresolved short codes remain separate from both
# empty strings and ordinary populated commentary.
course_and_type_comment_coverage = pd.read_sql_query(
    f"""
    WITH classified_rows AS (
        SELECT
            date,
            course,
            off,
            type,

            CASE
                WHEN comment = ''
                    THEN 'empty_string'

                WHEN comment IN (
                    '.',
                    '..',
                    '-',
                    ' -',
                    '/',
                    'A',
                    'B',
                    'V',
                    '1'
                )
                    THEN 'placeholder_or_unresolved_code'

                ELSE 'other_populated_text'
            END AS comment_state
        FROM data
        WHERE {DATA_ROW_PREDICATE}
    ),
    race_coverage AS (
        SELECT
            date,
            course,
            off,

            -- Race type should be constant within a provisional race.
            -- MAX(type) is used here only to carry the source value forward.
            MAX(type) AS race_type,

            COUNT(*) AS stored_runner_rows,

            SUM(comment_state = 'empty_string')
                AS empty_runner_rows,

            SUM(
                comment_state = 'placeholder_or_unresolved_code'
            ) AS placeholder_or_code_rows,

            SUM(
                comment_state = 'other_populated_text'
            ) AS other_populated_rows
        FROM classified_rows
        GROUP BY date, course, off
    )
    SELECT
        course,

        COUNT(*) AS provisional_races,
        SUM(stored_runner_rows) AS stored_runner_rows,

        SUM(empty_runner_rows) AS empty_runner_rows,
        SUM(placeholder_or_code_rows)
            AS placeholder_or_code_rows,
        SUM(other_populated_rows) AS other_populated_rows,

        ROUND(
            100.0 * SUM(other_populated_rows)
            / SUM(stored_runner_rows),
            2
        ) AS other_populated_runner_percentage,

        SUM(empty_runner_rows = stored_runner_rows)
            AS races_all_empty,

        SUM(other_populated_rows = stored_runner_rows)
            AS races_all_other_populated,

        SUM(
            empty_runner_rows > 0
            AND other_populated_rows > 0
        ) AS races_mixed_empty_and_populated,

        SUM(placeholder_or_code_rows > 0)
            AS races_with_placeholder_or_code
    FROM race_coverage
    GROUP BY course
    ORDER BY stored_runner_rows DESC, course
    """,
    connection,
)

# Summarise the same coverage states by source-presented race type.
#
# This tests whether missingness is associated with broad racing-code or
# race-description families, without assuming that every source `type` value
# is already semantically governed.
race_type_comment_coverage = pd.read_sql_query(
    f"""
    WITH classified_rows AS (
        SELECT
            date,
            course,
            off,
            type,

            CASE
                WHEN comment = ''
                    THEN 'empty_string'

                WHEN comment IN (
                    '.',
                    '..',
                    '-',
                    ' -',
                    '/',
                    'A',
                    'B',
                    'V',
                    '1'
                )
                    THEN 'placeholder_or_unresolved_code'

                ELSE 'other_populated_text'
            END AS comment_state
        FROM data
        WHERE {DATA_ROW_PREDICATE}
    ),
    race_coverage AS (
        SELECT
            date,
            course,
            off,
            MAX(type) AS race_type,

            COUNT(*) AS stored_runner_rows,

            SUM(comment_state = 'empty_string')
                AS empty_runner_rows,

            SUM(
                comment_state = 'placeholder_or_unresolved_code'
            ) AS placeholder_or_code_rows,

            SUM(
                comment_state = 'other_populated_text'
            ) AS other_populated_rows
        FROM classified_rows
        GROUP BY date, course, off
    )
    SELECT
        quote(race_type) AS quoted_race_type,

        COUNT(*) AS provisional_races,
        SUM(stored_runner_rows) AS stored_runner_rows,

        SUM(empty_runner_rows) AS empty_runner_rows,
        SUM(placeholder_or_code_rows)
            AS placeholder_or_code_rows,
        SUM(other_populated_rows) AS other_populated_rows,

        ROUND(
            100.0 * SUM(other_populated_rows)
            / SUM(stored_runner_rows),
            2
        ) AS other_populated_runner_percentage,

        SUM(empty_runner_rows = stored_runner_rows)
            AS races_all_empty,

        SUM(other_populated_rows = stored_runner_rows)
            AS races_all_other_populated,

        SUM(
            empty_runner_rows > 0
            AND other_populated_rows > 0
        ) AS races_mixed_empty_and_populated,

        SUM(placeholder_or_code_rows > 0)
            AS races_with_placeholder_or_code
    FROM race_coverage
    GROUP BY race_type
    ORDER BY stored_runner_rows DESC, quoted_race_type
    """,
    connection,
)

# Show the courses with the largest all-empty and mixed-coverage populations.
#
# Rankings by raw race count and by percentage answer different questions:
# one identifies the largest contribution to missingness, while the other
# identifies source labels where coverage is proportionally weakest.
course_missingness_focus = course_and_type_comment_coverage.loc[
    lambda frame:
        (frame["races_all_empty"] > 0)
        | (frame["races_mixed_empty_and_populated"] > 0)
].copy()

largest_all_empty_course_populations = (
    course_missingness_focus
    .sort_values(
        ["races_all_empty", "stored_runner_rows", "course"],
        ascending=[False, False, True],
    )
    .head(30)
)

lowest_course_coverage = (
    course_missingness_focus.loc[
        course_missingness_focus["stored_runner_rows"] >= 100
    ]
    .sort_values(
        [
            "other_populated_runner_percentage",
            "stored_runner_rows",
            "course",
        ],
        ascending=[True, False, True],
    )
    .head(30)
)

display(race_type_comment_coverage)
display(largest_all_empty_course_populations)
display(lowest_course_coverage)

,quoted_race_type,provisional_races,stored_runner_rows,empty_runner_rows,placeholder_or_code_rows,other_populated_rows,other_populated_runner_percentage,races_all_empty,races_all_other_populated,races_mixed_empty_and_populated,races_with_placeholder_or_code
0,'Flat',126391,1268229,291145,216,976868,77.03,18007,98536,9820,48
1,'Hurdle',35462,358441,31589,13,326839,91.18,2022,32442,997,7
2,'Chase',22547,179645,17445,9,162191,90.28,1510,20550,486,3
3,'NH Flat',4643,44970,215,0,44755,99.52,21,4620,2,0


,course,provisional_races,stored_runner_rows,empty_runner_rows,placeholder_or_code_rows,other_populated_rows,other_populated_runner_percentage,races_all_empty,races_all_other_populated,races_mixed_empty_and_populated,races_with_placeholder_or_code
3,Chantilly (FR),4140,48744,44818,3,3923,8.05,2821,267,1052,3
4,Deauville (FR),4051,48618,44342,2,4274,8.79,2714,286,1051,2
10,Auteuil (FR),3147,32708,30584,15,2109,6.45,2213,99,833,3
12,Saint-Cloud (FR),2558,28412,26122,5,2285,8.04,1679,160,718,1
17,Longchamp (FR),1941,20911,16648,4,4259,20.37,1071,407,463,4
72,Compiegne (FR),992,9347,8909,1,437,4.68,691,0,301,1
59,Maisons-Laffitte (FR),985,10770,9689,0,1081,10.04,571,62,352,0
95,Gulfstream Park (USA),745,6541,5873,75,593,9.07,502,34,197,12
100,Santa Anita (USA),828,6071,4069,0,2002,32.98,417,203,208,0
114,Woodbine (CAN),434,3519,2854,0,665,18.90,302,60,72,0


,course,provisional_races,stored_runner_rows,empty_runner_rows,placeholder_or_code_rows,other_populated_rows,other_populated_runner_percentage,races_all_empty,races_all_other_populated,races_mixed_empty_and_populated,races_with_placeholder_or_code
154,Hipodromo Chile (CHI),93,1152,1152,0,0,0.00,93,0,0,0
206,Fonner Park (USA),68,523,523,0,0,0.00,68,0,0,0
232,Belmont Park (Perth) (AUS),31,319,319,0,0,0.00,31,0,0,0
273,Morioka (JPN),19,195,195,0,0,0.00,19,0,0,0
275,Funabashi (JPN),20,193,193,0,0,0.00,20,0,0,0
288,San Isidro,11,155,155,0,0,0.00,11,0,0,0
293,Urawa (JPN),13,129,129,0,0,0.00,13,0,0,0
297,Gavea,9,118,118,0,0,0.00,9,0,0,0
306,NAGOYA (JPN),10,100,100,0,0,0.00,10,0,0,0
137,Will Rogers Downs (USA),178,1661,1660,0,1,0.06,177,0,1,0


### Course and race-type coverage findings

Comment availability varies sharply by race type and source-presented course.

## Race-type coverage

Across the four source-presented race types:

- **Flat:** 77.03% ordinary populated runner coverage;
- **Hurdle:** 91.18%;
- **Chase:** 90.28%;
- **NH Flat:** 99.52%.

Flat racing therefore accounts for most missing commentary:

- **291,145 empty Flat runner rows**;
- **18,007 all-empty Flat races**;
- **9,820 Flat races with mixed empty and populated coverage**.

By comparison:

- Hurdle contains **31,589 empty runner rows**;
- Chase contains **17,445**;
- NH Flat contains only **215**.

The contrast is too large to treat overall comment coverage as a uniform source property. Comment availability depends materially on race type, although race type is also correlated with jurisdiction and source feed.

NH Flat coverage is almost complete, while Flat coverage is much weaker. This does not establish that jump-racing comments are inherently better; much of the difference is likely explained by the database’s heavy overseas Flat population.

## Course concentration

The largest all-empty and partial-coverage populations are concentrated at overseas courses.

French examples include:

- Chantilly (FR): **8.05%** ordinary populated runner coverage;
- Deauville (FR): **8.79%**;
- Auteuil (FR): **6.45%**;
- Saint-Cloud (FR): **8.04%**;
- Compiegne (FR): **4.68%**;
- Longchamp (FR): **20.37%**.

Other weakly covered examples include:

- Gulfstream Park (USA): **9.07%**;
- Caulfield (AUS): **10.87%**;
- Randwick (AUS): **11.27%**;
- Nakayama (JPN): **2.64%**;
- Tokyo (JPN): **3.45%**;
- San Isidro (ARG): **1.66%**;
- Palermo (ARG): **1.08%**.

Several course labels have effectively no ordinary commentary:

- Hipodromo Chile (CHI): **0.00%**;
- Fonner Park (USA): **0.00%**;
- Belmont Park (Perth) (AUS): **0.00%**;
- Morioka (JPN): **0.00%**;
- Funabashi (JPN): **0.00%**;
- Urawa (JPN): **0.00%**;
- NAGOYA (JPN): **0.00%**.

Other large populations are nearly empty:

- Will Rogers Downs (USA): **0.06%**;
- Club Hipico de Santiago (CHI): **0.19%**;
- Cidade Jardim (BRZ): **0.41%**;
- Monterrico (PER): **0.42%**;
- Trentham (NZ): **0.45%**.

This is not random missingness. It is strongly associated with particular overseas feeds and source labels.

## Governed course identity

Course-label variation has already been resolved through the governed course-identity work completed in the previous notebook.

The source-presented labels in this output must therefore be treated as lineage values rather than unresolved course identities. Jurisdiction-level and venue-level summaries should use the existing governed course reference rather than re-investigating or informally merging labels inside this notebook.

This notebook will not reopen course identity unless the comment analysis exposes a genuinely unmatched source label or a contradiction in the governed reference.

## Interpretation

The main finding is:

> Missing comment data is structurally concentrated in overseas Flat racing and particular course feeds rather than spread evenly across the database.

This creates a serious selection issue for any analysis based on comment text.

A text-derived feature such as:

- running style;
- interference;
- jumping error;
- pace position;
- finishing effort;
- excuse;
- eyecatcher status;

would be observed much more often in British and Irish racing than in many overseas populations.

Absence of a phrase cannot therefore be interpreted as absence of the underlying event unless the comment itself is known to be substantively populated.

Current implications:

- comment-based analysis requires an explicit comment-availability filter;
- empty strings and placeholders must be separated from substantive comments;
- comparisons across jurisdictions or race types require coverage reporting;
- low-coverage course populations must not be treated as negative observations;
- jurisdiction summaries must use the existing governed course-identity reference.

## 10. Comment coverage by governed candidate jurisdiction

Comment availability may differ materially between racing jurisdictions, but jurisdiction must be derived through the settled Notebook 04 implementation rather than inferred informally from course text.

The governing function is not suitable for row-wise application across all 189,043 provisional races. Most course labels have a jurisdiction that is independent of individual race context. Only the bounded `Ascot` and `Newcastle` collision rules require date, type and race-name information.

This stage therefore:

- derives jurisdiction once for each distinct ordinary course label;
- processes only `Ascot` and `Newcastle` at provisional-race level;
- joins the compact mapping back to race-level comment coverage;
- fails if any race remains unresolved;
- summarises empty, placeholder/code and other populated comment rows by governed candidate jurisdiction.

This preserves the settled jurisdiction rules while avoiding a large Series-returning row-wise operation.

In [12]:
# Make the repository's src-layout package importable from the notebook.
import sys

source_package_path = str(PROJECT_ROOT / "src")

if source_package_path not in sys.path:
    sys.path.insert(0, source_package_path)

from inside_rails.course_jurisdiction import (
    derive_candidate_race_jurisdiction,
)

# Build one compact row per provisional race with comment-state counts.
race_comment_coverage = pd.read_sql_query(
    f"""
    WITH classified_rows AS (
        SELECT
            date,
            course,
            off,
            type,
            race_name,
            CASE
                WHEN comment = ''
                    THEN 'empty_string'
                WHEN comment IN (
                    '.',
                    '..',
                    '-',
                    ' -',
                    '/',
                    'A',
                    'B',
                    'V',
                    '1'
                )
                    THEN 'placeholder_or_unresolved_code'
                ELSE 'other_populated_text'
            END AS comment_state
        FROM data
        WHERE {DATA_ROW_PREDICATE}
    )
    SELECT
        date,
        course,
        off,
        MAX(type) AS type,
        MAX(race_name) AS race_name,
        COUNT(*) AS stored_runner_rows,
        SUM(comment_state = 'empty_string')
            AS empty_runner_rows,
        SUM(
            comment_state = 'placeholder_or_unresolved_code'
        ) AS placeholder_or_code_rows,
        SUM(
            comment_state = 'other_populated_text'
        ) AS other_populated_rows
    FROM classified_rows
    GROUP BY date, course, off
    """,
    connection,
)

# Most jurisdictions depend only on the raw course label. Reduce those decisions
# to one row per distinct course instead of applying Python across every race.
context_dependent_courses = {"Ascot", "Newcastle"}

ordinary_course_labels = (
    race_comment_coverage.loc[
        ~race_comment_coverage["course"].isin(
            context_dependent_courses
        ),
        ["course"],
    ]
    .drop_duplicates()
    .sort_values("course")
    .reset_index(drop=True)
)

# Supply neutral values for fields unused by ordinary course rules.
ordinary_course_decisions = ordinary_course_labels.assign(
    date="",
    type="",
    race_name="",
)

ordinary_course_decisions[
    [
        "candidate_jurisdiction",
        "jurisdiction_evidence",
    ]
] = ordinary_course_decisions.apply(
    derive_candidate_race_jurisdiction,
    axis=1,
)

ordinary_course_decisions = ordinary_course_decisions[
    [
        "course",
        "candidate_jurisdiction",
        "jurisdiction_evidence",
    ]
]

# Join the compact ordinary-course mapping back to provisional races.
race_comment_coverage = race_comment_coverage.merge(
    ordinary_course_decisions,
    how="left",
    on="course",
    validate="many_to_one",
)

# Apply the two documented race-context collision rules vectorially.
for collision_course in sorted(context_dependent_courses):
    course_mask = (
        race_comment_coverage["course"] == collision_course
    )

    australian_race_mask = (
        course_mask
        & (race_comment_coverage["date"] >= "2025-10-15")
        & (race_comment_coverage["type"] == "Flat")
        & race_comment_coverage["race_name"]
            .str.contains(r"\(Turf\)", na=False)
    )

    british_race_mask = course_mask & ~australian_race_mask

    race_comment_coverage.loc[
        australian_race_mask,
        "candidate_jurisdiction",
    ] = "Australia"

    race_comment_coverage.loc[
        british_race_mask,
        "candidate_jurisdiction",
    ] = "Great Britain"

    race_comment_coverage.loc[
        course_mask,
        "jurisdiction_evidence",
    ] = "race_context_course_collision_rule"

# Stop before aggregation if the settled rules do not cover the full race set.
unresolved_jurisdiction_races = race_comment_coverage.loc[
    race_comment_coverage["candidate_jurisdiction"].isna()
    | (
        race_comment_coverage["candidate_jurisdiction"]
        == "unresolved"
    )
].copy()

if not unresolved_jurisdiction_races.empty:
    raise ValueError(
        "The governed jurisdiction rules left "
        f"{len(unresolved_jurisdiction_races):,} provisional races "
        "unresolved."
    )

# Classify race-level comment coverage patterns before jurisdiction aggregation.
race_comment_coverage = race_comment_coverage.assign(
    race_all_empty=lambda frame:
        frame["empty_runner_rows"]
        == frame["stored_runner_rows"],

    race_all_other_populated=lambda frame:
        frame["other_populated_rows"]
        == frame["stored_runner_rows"],

    race_mixed_empty_and_populated=lambda frame:
        (frame["empty_runner_rows"] > 0)
        & (frame["other_populated_rows"] > 0),

    race_with_placeholder_or_code=lambda frame:
        frame["placeholder_or_code_rows"] > 0,
)

jurisdiction_comment_coverage = (
    race_comment_coverage
    .groupby(
        "candidate_jurisdiction",
        as_index=False,
    )
    .agg(
        provisional_races=("course", "size"),
        stored_runner_rows=("stored_runner_rows", "sum"),
        empty_runner_rows=("empty_runner_rows", "sum"),
        placeholder_or_code_rows=(
            "placeholder_or_code_rows",
            "sum",
        ),
        other_populated_rows=("other_populated_rows", "sum"),
        races_all_empty=("race_all_empty", "sum"),
        races_all_other_populated=(
            "race_all_other_populated",
            "sum",
        ),
        races_mixed_empty_and_populated=(
            "race_mixed_empty_and_populated",
            "sum",
        ),
        races_with_placeholder_or_code=(
            "race_with_placeholder_or_code",
            "sum",
        ),
    )
)

jurisdiction_comment_coverage[
    "other_populated_runner_percentage"
] = (
    100.0
    * jurisdiction_comment_coverage["other_populated_rows"]
    / jurisdiction_comment_coverage["stored_runner_rows"]
).round(2)

jurisdiction_comment_coverage = (
    jurisdiction_comment_coverage
    .sort_values(
        [
            "stored_runner_rows",
            "candidate_jurisdiction",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

# Reconcile against the governed source population before interpretation.
jurisdiction_reconciliation = pd.DataFrame(
    {
        "measure": [
            "ordinary course decisions evaluated in Python",
            "provisional races",
            "stored runner rows",
            "unresolved jurisdiction races",
            "jurisdictions represented",
        ],
        "value": [
            len(ordinary_course_decisions),
            len(race_comment_coverage),
            race_comment_coverage[
                "stored_runner_rows"
            ].sum(),
            len(unresolved_jurisdiction_races),
            race_comment_coverage[
                "candidate_jurisdiction"
            ].nunique(),
        ],
    }
)

display(jurisdiction_reconciliation)
display(jurisdiction_comment_coverage)

,measure,value
0,ordinary course decisions evaluated in Python,526
1,provisional races,189043
2,stored runner rows,1851285
3,unresolved jurisdiction races,0
4,jurisdictions represented,36


,candidate_jurisdiction,provisional_races,stored_runner_rows,empty_runner_rows,placeholder_or_code_rows,other_populated_rows,races_all_empty,races_all_other_populated,races_mixed_empty_and_populated,races_with_placeholder_or_code,other_populated_runner_percentage
0,Great Britain,111634,984757,0,0,984757,0,111634,0,0,100.00
1,Ireland,30783,356950,0,0,356950,0,30783,0,0,100.00
2,France,20533,228332,207677,48,20607,13328,1359,5842,26,9.03
3,Hong Kong,7481,90975,285,0,90690,19,7451,11,0,99.69
4,United States,6105,50145,37312,84,12749,3070,1134,1888,13,25.42
5,Australia,4059,46156,41850,10,4296,1817,76,2164,3,9.31
6,United Arab Emirates,2451,27890,98,27,27765,6,2442,1,2,99.55
7,Japan,1559,22551,21955,15,581,1119,0,439,3,2.58
8,Germany,731,6368,2127,0,4241,105,457,169,0,66.60
9,South Africa,470,5727,5469,9,249,285,1,183,1,4.35


### Jurisdiction coverage findings

The governed jurisdiction mapping partitions the full source population successfully:

- 189,043 provisional races;
- 1,851,285 stored runner rows;
- 36 candidate jurisdictions;
- no unresolved jurisdiction assignments.

Comment availability is strongly jurisdiction-dependent.

Great Britain and Ireland are complete in this source:

- Great Britain: 984,757 of 984,757 runner rows contain other populated text;
- Ireland: 356,950 of 356,950 runner rows contain other populated text;
- every provisional race in both jurisdictions is fully populated;
- neither jurisdiction contains an empty comment row or a profiled placeholder/code row.

Several other feeds are also close to complete:

- Hong Kong: 99.69% other populated text;
- United Arab Emirates: 99.55%;
- Guernsey: 100%, although only 80 runner rows are present.

The major overseas Flat feeds show very different coverage:

- France: 9.03%;
- United States: 25.42%;
- Australia: 9.31%;
- Japan: 2.58%;
- South Africa: 4.35%.

For France, the United States and Australia, missingness occurs at both whole-race and within-race level. France alone contains:

- 13,328 all-empty races;
- 5,842 races mixing empty and populated runner comments;
- only 1,359 races where every runner has other populated text.

This confirms that source-wide comment missingness is not random and should not be described through a single overall percentage. It largely reflects jurisdiction-specific feed behaviour.

The field is therefore highly complete for British and Irish racing, nearly complete for some other jurisdictions, and sparse or structurally selective in several overseas feeds.

Placeholder and unresolved-code values are numerically minor relative to ordinary empty strings. Their presence does not explain the principal coverage differences.

Database and analytical consequences:

- preserve the candidate jurisdiction and jurisdiction-evidence fields alongside any comment-derived feature;
- report coverage by jurisdiction before comparing comment-derived measures;
- do not interpret an empty comment as evidence that no incident, tactic or performance detail existed;
- do not compare jurisdictions directly without accounting for their materially different recording regimes;
- treat British and Irish comment analysis as a substantially more complete population than analysis of most overseas feeds.

## 11. Compare comment form across selected jurisdictions

Coverage alone does not establish that populated comments have the same meaning or structure across jurisdictions.

This stage will compare a bounded, reproducible sample from jurisdictions with materially different coverage:

- Great Britain;
- Ireland;
- France;
- United States;
- Australia;
- Hong Kong;
- United Arab Emirates;
- Germany;
- Italy.

The purpose is exploratory rather than classificatory. It will test whether populated values differ in:

- typical text length;
- punctuation density;
- use of terminal parenthetical material;
- apparent sentence or clause structure;
- repeated short-template behaviour;
- general resemblance to British and Irish in-running prose.

To keep the operation proportionate:

- jurisdiction is reused from the already governed race-level table;
- only a fixed number of qualifying races is sampled per jurisdiction;
- only comments attached to those sampled races are read from SQLite;
- no source-wide Python row function or full runner-level merge is used;
- raw comment text remains unchanged.

The output will support a descriptive judgement about feed form. It will not establish who authored the comments, whether translations occurred, or whether one jurisdiction’s comments are intrinsically better.

## 11. Temporary analytics database and cross-jurisdiction comment inspection

The source Raceform database remains immutable and read-only.

Notebook 21 now creates a separate temporary SQLite database for transient analytical work. This permits temporary tables and SQL joins without writing to the source file. The temporary database exists only for the current notebook session and is removed when its temporary directory is cleaned up.

This stage uses that database to:

- store a bounded sample of provisional race keys;
- attach the source database in read-only immutable mode;
- retrieve only the corresponding populated comments;
- compare comment form across selected jurisdictions;
- print complete raw comments for direct human inspection.

The sampled text remains unchanged. Length and punctuation measurements are descriptive only and do not constitute a parser or quality score.

In [16]:
import sqlite3
import tempfile
from pathlib import Path
from urllib.parse import quote

# Close and remove any earlier temporary analytics database created during the
# current notebook session before replacing it.
if "analytics_connection" in globals():
    analytics_connection.close()

if "analytics_temp_directory" in globals():
    analytics_temp_directory.cleanup()

# Create a writable temporary directory and SQLite database outside the
# immutable source-data location.
analytics_temp_directory = tempfile.TemporaryDirectory(
    prefix="inside_rails_notebook_21_"
)

analytics_database_path = (
    Path(analytics_temp_directory.name)
    / "notebook_21_analytics.sqlite"
)

analytics_connection = sqlite3.connect(
    analytics_database_path
)

# Resolve the source database path from the existing governed read-only
# connection rather than duplicating or guessing its repository location.
source_database_entries = connection.execute(
    "PRAGMA database_list"
).fetchall()

source_database_path = next(
    Path(database_path).resolve()
    for _, database_name, database_path in source_database_entries
    if database_name == "main"
)

# Attach the source file to the writable analytics database explicitly in
# read-only immutable mode. Temporary tables will be created only in the
# analytics database.
encoded_source_path = quote(
    source_database_path.as_posix(),
    safe="/:",
)

source_database_uri = (
    f"file:{encoded_source_path}?mode=ro&immutable=1"
)

analytics_connection.execute(
    "ATTACH DATABASE ? AS source_data",
    (source_database_uri,),
)

# Confirm that both databases are visible and that their roles are explicit.
analytics_database_register = pd.DataFrame(
    analytics_connection.execute(
        "PRAGMA database_list"
    ).fetchall(),
    columns=[
        "sequence",
        "database_name",
        "database_path",
    ],
)

analytics_database_register["intended_role"] = (
    analytics_database_register["database_name"].map(
        {
            "main": "writable temporary analytics state",
            "source_data": "read-only immutable source",
        }
    )
)

display(analytics_database_register)

,sequence,database_name,database_path,intended_role
0,0,main,/tmp/inside_rails_notebook_21_qvtnv13x/noteboo...,writable temporary analytics state
1,2,source_data,/home/rob/Documents/inside-rails-horse-racing/...,read-only immutable source


In [17]:
import re

# Compare jurisdictions with materially different source coverage while
# keeping the inspection sample bounded and reproducible.
selected_jurisdictions = [
    "Great Britain",
    "Ireland",
    "France",
    "United States",
    "Australia",
    "Hong Kong",
    "United Arab Emirates",
    "Germany",
    "Italy",
]

eligible_races = race_comment_coverage.loc[
    race_comment_coverage["candidate_jurisdiction"].isin(
        selected_jurisdictions
    )
    & (race_comment_coverage["other_populated_rows"] > 0),
    [
        "date",
        "course",
        "off",
        "candidate_jurisdiction",
    ],
].copy()

# Randomise once with a fixed seed, then retain at most 50 eligible races per
# jurisdiction without using row-wise or group-wise Python functions.
random_order = eligible_races.sample(
    frac=1,
    random_state=21,
).index

eligible_races["sampling_order"] = pd.Series(
    range(len(random_order)),
    index=random_order,
)

sampled_race_keys = (
    eligible_races
    .sort_values(
        [
            "candidate_jurisdiction",
            "sampling_order",
        ]
    )
    .groupby(
        "candidate_jurisdiction",
        group_keys=False,
    )
    .head(50)
    .drop(columns="sampling_order")
    .reset_index(drop=True)
)

sampled_race_counts = (
    sampled_race_keys
    .groupby(
        "candidate_jurisdiction",
        as_index=False,
    )
    .size()
    .rename(columns={"size": "sampled_races"})
)

# Store the small key table in the writable temporary analytics database.
analytics_connection.execute(
    "DROP TABLE IF EXISTS sampled_comment_race_keys"
)

sampled_race_keys.to_sql(
    "sampled_comment_race_keys",
    analytics_connection,
    index=False,
    if_exists="replace",
)

# Join temporary keys to the attached immutable source inside SQLite and read
# back only populated, non-placeholder comments from sampled races.
jurisdiction_comment_sample = pd.read_sql_query(
    """
    SELECT
        keys.candidate_jurisdiction,
        source.date,
        source.course,
        source.off,
        source.horse,
        source.comment
    FROM sampled_comment_race_keys AS keys
    INNER JOIN source_data.data AS source
        ON source.date = keys.date
        AND source.course = keys.course
        AND source.off = keys.off
    WHERE source.rowid <> 1
      AND source.comment <> ''
      AND source.comment NOT IN (
          '.',
          '..',
          '-',
          ' -',
          '/',
          'A',
          'B',
          'V',
          '1'
      )
    """,
    analytics_connection,
)

# Derive descriptive surface-form measurements while preserving the complete
# raw comment in its original column.
terminal_parenthetical_pattern = re.compile(
    r"\([^()]*\)\s*$"
)

jurisdiction_comment_sample = (
    jurisdiction_comment_sample
    .assign(
        comment_length=lambda frame:
            frame["comment"].str.len(),

        word_count=lambda frame:
            frame["comment"].str.split().str.len(),

        comma_count=lambda frame:
            frame["comment"].str.count(","),

        semicolon_count=lambda frame:
            frame["comment"].str.count(";"),

        terminal_parenthetical=lambda frame:
            frame["comment"].str.contains(
                terminal_parenthetical_pattern,
                na=False,
            ),

        contains_sentence_stop=lambda frame:
            frame["comment"].str.contains(
                r"[.!?]",
                regex=True,
                na=False,
            ),
    )
)

jurisdiction_comment_form = (
    jurisdiction_comment_sample
    .groupby(
        "candidate_jurisdiction",
        as_index=False,
    )
    .agg(
        sampled_comments=("comment", "size"),
        distinct_comments=("comment", "nunique"),
        median_characters=("comment_length", "median"),
        mean_characters=("comment_length", "mean"),
        median_words=("word_count", "median"),
        mean_commas=("comma_count", "mean"),
        mean_semicolons=("semicolon_count", "mean"),
        terminal_parenthetical_comments=(
            "terminal_parenthetical",
            "sum",
        ),
        sentence_stop_comments=(
            "contains_sentence_stop",
            "sum",
        ),
    )
)

jurisdiction_comment_form[
    "terminal_parenthetical_percentage"
] = (
    100
    * jurisdiction_comment_form[
        "terminal_parenthetical_comments"
    ]
    / jurisdiction_comment_form["sampled_comments"]
).round(2)

jurisdiction_comment_form[
    "sentence_stop_percentage"
] = (
    100
    * jurisdiction_comment_form[
        "sentence_stop_comments"
    ]
    / jurisdiction_comment_form["sampled_comments"]
).round(2)

for column in [
    "mean_characters",
    "mean_commas",
    "mean_semicolons",
]:
    jurisdiction_comment_form[column] = (
        jurisdiction_comment_form[column].round(2)
    )

jurisdiction_comment_form = (
    sampled_race_counts
    .merge(
        jurisdiction_comment_form,
        on="candidate_jurisdiction",
        how="left",
        validate="one_to_one",
    )
    .sort_values("candidate_jurisdiction")
    .reset_index(drop=True)
)

# Select three deterministic examples per jurisdiction.
jurisdiction_comment_examples = (
    jurisdiction_comment_sample
    .sort_values(
        [
            "candidate_jurisdiction",
            "date",
            "course",
            "off",
            "horse",
        ]
    )
    .groupby(
        "candidate_jurisdiction",
        group_keys=False,
    )
    .head(3)
    [
        [
            "candidate_jurisdiction",
            "date",
            "course",
            "off",
            "horse",
            "comment",
        ]
    ]
    .reset_index(drop=True)
)

display(jurisdiction_comment_form)

# Print rather than tabulate the examples so Jupyter cannot truncate the raw
# comment text.
for jurisdiction, examples in (
    jurisdiction_comment_examples.groupby(
        "candidate_jurisdiction",
        sort=True,
    )
):
    print(f"\n{jurisdiction}")
    print("=" * len(jurisdiction))

    for example in examples.itertuples(index=False):
        print(
            f"\n{example.date} | {example.course} | "
            f"{example.off} | {example.horse}"
        )
        print(example.comment)

,candidate_jurisdiction,sampled_races,sampled_comments,distinct_comments,median_characters,mean_characters,median_words,mean_commas,mean_semicolons,terminal_parenthetical_comments,sentence_stop_comments,terminal_parenthetical_percentage,sentence_stop_percentage
0,Australia,50,97,97,113.0,116.38,23.0,0.0,0.01,1,0,1.03,0.00
1,France,50,129,129,128.0,129.96,25.0,0.0,0.00,3,0,2.33,0.00
2,Germany,50,338,337,112.0,115.07,23.0,0.0,0.00,0,0,0.00,0.00
3,Great Britain,50,483,483,113.0,121.76,23.0,0.0,0.03,403,1,83.44,0.21
4,Hong Kong,50,579,551,92.0,93.25,19.0,0.0,0.00,0,0,0.00,0.00
5,Ireland,50,559,555,132.0,137.03,26.0,0.0,0.00,433,1,77.46,0.18
6,Italy,50,402,399,106.0,108.30,21.0,0.0,0.00,4,0,1.00,0.00
7,United Arab Emirates,50,547,397,51.0,52.94,10.0,0.0,0.01,123,0,22.49,0.00
8,United States,50,193,192,110.0,111.37,22.0,0.0,0.00,0,0,0.00,0.00



Australia

2015-02-21 | Flemington (AUS) | 4:55 | Wandjina (AUS)
Soon led on outer - switched inside to rail after 2f - pressed on either side 2 1/2f out - ridden 1 1/2f out - headed inside final furlong - rallied gamely to regain lead 50yds out

2015-04-18 | Kembla Grange (AUS) | 4:26 | Subservient (AUS)
Chased along to lead after 1f - kicked clear 1 1/2f out - headed inside final furlong - no extra

2015-10-17 | Caulfield (AUS) | 7:40 | Fame Game (JPN)
Held up towards rear - dropped to rear 2f out - ridden and stayed on well final furlong - nearest finish

France

2015-03-29 | Machecoul (FR) | 2:00 | Important Time (IRE)
Took keen hold - held up behind leading group on outer - close 5th and every chance over 2f out - quickened to lead 1 1/2f out - driven clear approaching final furlong - easily

2015-04-15 | Chantilly (FR) | 1:05 | Attentif (GB)
Prominent on outer early - soon steadied and midfield in touch - ridden 2f out - bumped as winner switched out over 1f out - not quicken fi

### Cross-jurisdiction comment-form findings

The sampled populated comments are broadly the same kind of runner-level in-running prose across the selected jurisdictions.

Australia, France, Germany, Great Britain, Hong Kong, Ireland, Italy and the United States all contain recognisably similar descriptions of:

- early race position;
- changes in position;
- riding pressure;
- headway or weakening;
- mistakes, pace and finishing effort.

The examples do not suggest that the sparse French, Australian or United States coverage represents a fundamentally different field. Where comments are present, they resemble the same English-language descriptive racing prose used for British and Irish runners.

Typical comment length is also broadly comparable:

- Australia: median 113 characters;
- France: 128;
- Germany: 112;
- Great Britain: 113;
- Hong Kong: 92;
- Ireland: 132;
- Italy: 106;
- United States: 110.

This strengthens the interpretation that the principal overseas issue is **availability**, not field meaning.

The United Arab Emirates sample differs somewhat:

- median length is only 51 characters and 10 words;
- 547 sampled comments contain only 397 distinct values;
- several comments are short summary forms such as `Never better than mid-division`.

The UAE field still appears to describe race performance, but its feed contains shorter and more repeated formulations than most other selected jurisdictions.

Terminal parenthetical material is concentrated in Great Britain and Ireland:

- Great Britain: 83.44%;
- Ireland: 77.46%;
- United Arab Emirates: 22.49%;
- all other sampled jurisdictions: approximately 0–2%.

This supports the earlier finding that British and Irish comments frequently embed additional terminal information, including market-related or attributed material, while most overseas comments primarily contain the in-running narrative itself.

The punctuation measurements are less informative than expected. These comments are predominantly structured with spaced hyphens rather than commas, semicolons or full sentence punctuation. Near-zero comma and sentence-stop rates therefore reflect the source’s house style rather than absence of internal structure.

The cross-jurisdiction evidence supports the following semantic conclusion:

> When populated, `comment` is generally a runner-level English-language description of race position and performance. Coverage, detail and embedded terminal material vary by jurisdiction and feed, but the underlying field meaning remains broadly consistent.

Analytical consequences:

- populated overseas comments may be analysed under the same broad semantic interpretation as British and Irish comments;
- coverage differences must still be reported before cross-jurisdiction analysis;
- British and Irish terminal parenthetical content should not be assumed to exist in other feeds;
- short and repeated UAE formulations should remain distinguishable from longer narrative comments;
- punctuation-based sentence detection is unsuitable for this field without first accounting for its hyphen-delimited house style.

## 12. Provisional semantic and database decisions

The source-wide profiling, manual inspection and jurisdiction comparison now support a bounded interpretation of `comment`.

This stage records the decisions reached so far without implementing a parser or replacing the raw field.

The register distinguishes:

- confirmed field meaning;
- source-coverage limitations;
- unresolved short codes and placeholders;
- embedded terminal material;
- permitted database treatment;
- work explicitly deferred to a later study.

These are governance decisions for preserving and using the source field, not claims that every comment can already be parsed reliably.

In [20]:
# Record the field-level conclusions reached from Notebook 21 evidence.
# This table governs preservation and analytical use without authorising a
# speculative parser or destructive cleaning rule.
comment_semantic_decisions = pd.DataFrame(
    [
        {
            "decision_area": "Field meaning",
            "decision": (
                "Interpret populated substantive values as runner-level "
                "English-language descriptions of race position and performance."
            ),
            "status": "Confirmed",
            "evidence": (
                "Manual inspection across British, Irish and selected overseas "
                "jurisdictions showed consistent in-running narrative content."
            ),
        },
        {
            "decision_area": "Jurisdiction consistency",
            "decision": (
                "Use the same broad field meaning across jurisdictions when "
                "substantive text is present."
            ),
            "status": "Confirmed",
            "evidence": (
                "France, Germany, Australia, Italy, Hong Kong, the United States "
                "and the UAE contained recognisably comparable performance prose."
            ),
        },
        {
            "decision_area": "Coverage",
            "decision": (
                "Treat comment availability as jurisdiction- and feed-dependent "
                "rather than as a source-wide random missingness process."
            ),
            "status": "Confirmed",
            "evidence": (
                "Great Britain and Ireland were complete while several overseas "
                "feeds were sparse or selective."
            ),
        },
        {
            "decision_area": "Empty values",
            "decision": (
                "Preserve an empty string as source absence and do not interpret "
                "it as evidence that no notable race event occurred."
            ),
            "status": "Required",
            "evidence": (
                "Whole-race and mixed within-race missingness patterns varied "
                "materially by jurisdiction."
            ),
        },
        {
            "decision_area": "Short codes and placeholders",
            "decision": (
                "Preserve profiled short values separately from substantive prose "
                "until their source meanings are independently established."
            ),
            "status": "Required",
            "evidence": (
                "Values such as '.', 'A', 'B' and 'V' were rare, and the tested "
                "equipment interpretation was not supported."
            ),
        },
        {
            "decision_area": "Terminal parenthetical material",
            "decision": (
                "Recognise that British and Irish comments frequently contain "
                "embedded terminal information beyond the in-running narrative."
            ),
            "status": "Confirmed",
            "evidence": (
                "Terminal parentheticals appeared in 83.44% of sampled British "
                "comments and 77.46% of sampled Irish comments."
            ),
        },
        {
            "decision_area": "Market information",
            "decision": (
                "Retain market-related terminal text within the raw comment and "
                "do not yet extract it into permanent structured fields."
            ),
            "status": "Deferred",
            "evidence": (
                "Recognisable markers exist, but malformed, nested and attributed "
                "parentheticals prevent authorising a simple terminal parser."
            ),
        },
        {
            "decision_area": "Attributed reports",
            "decision": (
                "Preserve jockey, trainer and other attributed statements as part "
                "of the raw source comment."
            ),
            "status": "Required",
            "evidence": (
                "Manual examples showed attributed reports can appear immediately "
                "before market parentheticals and may contain their own brackets."
            ),
        },
        {
            "decision_area": "Comment parsing",
            "decision": (
                "Do not implement a general comment parser during this source-field "
                "study."
            ),
            "status": "Deferred",
            "evidence": (
                "The field combines hyphen-delimited narrative, reports, market "
                "markers and malformed or nested source structures."
            ),
        },
        {
            "decision_area": "Database storage",
            "decision": (
                "Preserve the exact raw comment and attach lineage, jurisdiction "
                "and any future derived assertions separately."
            ),
            "status": "Required",
            "evidence": (
                "Raw wording and source structure are analytically valuable and "
                "coverage regimes differ materially."
            ),
        },
        {
            "decision_area": "Future enrichment",
            "decision": (
                "Any externally sourced or translated commentary must be stored "
                "separately with provider, language, retrieval and join provenance."
            ),
            "status": "Deferred",
            "evidence": (
                "The present study establishes source limitations but does not "
                "establish recoverability, licensing or historical availability."
            ),
        },
    ]
)

display(comment_semantic_decisions)

,decision_area,decision,status,evidence
0,Field meaning,Interpret populated substantive values as runn...,Confirmed,"Manual inspection across British, Irish and se..."
1,Jurisdiction consistency,Use the same broad field meaning across jurisd...,Confirmed,"France, Germany, Australia, Italy, Hong Kong, ..."
2,Coverage,Treat comment availability as jurisdiction- an...,Confirmed,Great Britain and Ireland were complete while ...
3,Empty values,Preserve an empty string as source absence and...,Required,Whole-race and mixed within-race missingness p...
4,Short codes and placeholders,Preserve profiled short values separately from...,Required,"Values such as '.', 'A', 'B' and 'V' were rare..."
5,Terminal parenthetical material,Recognise that British and Irish comments freq...,Confirmed,Terminal parentheticals appeared in 83.44% of ...
6,Market information,Retain market-related terminal text within the...,Deferred,"Recognisable markers exist, but malformed, nes..."
7,Attributed reports,"Preserve jockey, trainer and other attributed ...",Required,Manual examples showed attributed reports can ...
8,Comment parsing,Do not implement a general comment parser duri...,Deferred,"The field combines hyphen-delimited narrative,..."
9,Database storage,Preserve the exact raw comment and attach line...,Required,Raw wording and source structure are analytica...


### Semantic decision summary

The decision register now separates what is established from what remains deliberately unresolved.

The confirmed interpretation is that `comment`, when substantively populated, is a runner-level English-language description of race position and performance. That meaning is broadly consistent across the inspected jurisdictions, even though availability differs sharply between feeds.

The most important limitation is therefore coverage rather than semantics:

- British and Irish comments are effectively complete in this source;
- several overseas jurisdictions are sparse or selective;
- empty values must remain source absence rather than being converted into a substantive category.

The field must be preserved exactly because it can contain several information types within one value:

- in-running narrative;
- attributed jockey or trainer reports;
- market-related terminal text;
- malformed or nested parenthetical structures;
- rare unresolved short codes or placeholders.

Notebook 21 does not authorise a general parser. Any future extraction must create separate derived assertions with lineage back to the untouched raw comment.

The database treatment is therefore:

> Preserve the exact source comment, preserve source absence and unresolved values explicitly, attach jurisdiction and lineage separately, and defer structured extraction until each embedded information type has a validated parsing rule.